# 第 05A 节　Trae 调试功能的使用

> **本节定位**：前面五章解决"写得出、跑得通"，本节解决"跑得不对怎么办"。调试不是排错的最后一招，而是**看清程序真面目**的第一手段——不用改代码、不用加 `print`，就能让程序停在任意一行，逐句看每个变量怎么变。
>
> **知识点范围**：Trae 调试环境的准备；行断点与启动调试（`F9` / `F5`）；逐过程（`F10`）；步入（`F11`）；步出（`Shift+F11`）；变量面板、悬停求值与监视表达式；调用堆栈；条件断点与日志点。**不引入新语法**，本节代码只用第 1—5 章的列表、元组、集合、字典、判断、循环、字符串格式化与推导式。
>
> **本节约定**：为了演示"步入 / 步出"，示例代码中出现少量 `def` 函数定义——**只需看懂"哪里被调用、能进得去"，函数语法的系统讲解在第 6 章**，本节不要求独立编写函数。所有预设 bug 都是**"不报错、只算错"的逻辑错误**，这正是调试器的主场。
>
> **建议课时**：4 学时（2 次课），上机为主。

> **先修知识**：第 01—05 章全部语法（判断、循环、列表、元组、集合、字典、字符串格式化）；本机已安装 Trae 并选定 Python 解释器（准备步骤见 5A.1）。本节不要求会写函数，只要求看懂 `def` 的调用位置。
>
> **学习目标**：完成本节后，你能够：
>
> 1. 在 `示例5A1` 上设置行断点并按 `F5` 启动，使程序停在断点行，读出当前行号与已赋值变量；
> 2. 用 `F10` 逐过程执行完 1 个循环，记录每轮计数器取值，并与直接运行的输出比对；
> 3. 用 `F11` 进入被调用函数、用 `Shift+F11` 返回调用处，写出进入与返回的行号；
> 4. 添加 1 个监视表达式，结合调用堆栈写出「现象—错误值—正确值」三列定位表；
> 5. 用条件断点让程序只在第 N 条记录处停下，并说明日志点与 `print` 的输出对应关系。

---

## 本节学习路径

| 阶段 | 内容 | 编号 | 作用 |
|---|---|---|---|
| 第一部分 | 知识讲解 | 知识点 5A.1 — 5A.7 | 七个高频调试技能，逐个「集中讲解 → 示例代码 → 练习代码」 |
| 第二部分 | 代码示范 | 案例 5A-1、案例 5A-2 | 信号配时（3 个 bug）/ 早高峰统计（4 个 bug） |
| 第三部分 | 布置练习 | 练习 5A-1、练习 5A-2 | ETC 计费（3 个 bug）/ 分断面告警（4 个 bug） |
| 第四部分 | 习题答案 | 答案 5A-1、答案 5A-2 | bug 清单、修复代码、定位路径与评分要点 |

**知识点与代码对照表（调试技能版，额外保留「快捷键」列）**：

| 知识点 | 技能 | 快捷键 | 示例代码 | 练习代码 | 后续用在哪个 bug |
|---|---|---|---|---|---|
| 5A.1 | 断点与启动调试 | `F9` / `F5` | `示例5A1_断点与启动调试.py` | `练习5A1_断点与启动调试.py` | 全部 bug 的入口 |
| 5A.2 | 逐过程（单步跳过） | `F10` | `示例5A2_逐过程执行.py` | `练习5A2_逐过程执行.py` | 案例 5A-1 的 bug 1 |
| 5A.3 | 步入 | `F11` | `示例5A3_步入函数.py` | `练习5A3_步入函数.py` | 案例 5A-1 的 bug 2 |
| 5A.4 | 步出 | `Shift+F11` | `示例5A4_步出函数.py` | `练习5A4_步出函数.py` | 案例 5A-2 的 bug 2 |
| 5A.5 | 查看变量与方法 | 悬停 / 监视 | `示例5A5_变量与监视.py` | `练习5A5_变量与监视.py` | 案例 5A-1 的 bug 3 |
| 5A.6 | 调用堆栈 | «调用堆栈» 面板 | `示例5A6_调用堆栈.py` | `练习5A6_调用堆栈.py` | 练习 5A-2 的 bug 2 |
| 5A.7 | 条件断点与日志点 | 右键断点 | `示例5A7_条件断点与日志点.py` | `练习5A7_条件断点与日志点.py` | 案例 5A-2 的 bug 1 / 4 |

## 教学提示（教师用）

- **建议课时**：4 学时。第 1 次课讲 5A.1—5A.4（把"停下来、走一步、进出函数"练成肌肉记忆），第 2 次课讲 5A.5—5A.7 并完成两个案例。
- **讲解主线**：调试只有一个核心动作——**让程序停下来**。停在哪（断点）、怎么走（F10/F11/Shift+F11）、看什么（变量/监视/堆栈）、什么时候停（条件断点），四问对应本节七个技能。
- **最重要的一句话**：`print` 只能看到你**想到要打印**的东西，调试器能看到你**没想到**的东西。案例 5A-1 的 bug 1 就是"以为 `y_total` 在累加，其实每轮都被清零"——`print` 打在循环外永远看不出来。
- **学生常见卡点**：① 断点设了但按 `Ctrl+F5`（直接运行）而没按 `F5`（启动调试），断点当然不生效；② 断点变成灰色空心圆，多半是没选解释器或文件路径不匹配；③ 把 `F10` 和 `F11` 弄反，一按就"跳进"标准库源码，看得云里雾里——此时按 `Shift+F11` 立刻回来；④ 只盯着变量面板最外层，忘了展开字典 / 列表。
- **课堂纪律**：先让学生**说出预期结果**，再运行看实际结果，最后才准开调试器。没有预期的调试是"逛代码"。
- **课堂活动建议**：两人一组，一人念"我现在按 F10，你认为 `total` 会变成多少"，另一人操作并核对；错一次交换角色。这个"预测—验证"环节比讲十遍快捷键有效。

## 课前准备：把 Trae 的调试环境接通（10 分钟）

| 步骤 | 操作 | 验收标准 |
|---|---|---|
| 1 | 安装 Trae（`trae.ai` 或 `trae.cn` 下载对应系统版本），登录账号 | 能新建 / 打开文件夹 |
| 2 | Trae 菜单 **文件 → 打开文件夹**，选中本课程目录 `05A_Trae调试功能` | 左侧「资源管理器」能看到 4 个案例 / 练习 `.py` 与 7 组示例、练习 `.py` |
| 3 | 命令面板（`Ctrl+Shift+P` / `Cmd+Shift+P`）→ 输入 **Python: Select Interpreter** → 选择本机 Python 3 | 右下角状态栏显示解释器版本 |
| 4 | 打开任一 `.py`，按 **`F5`**，首次调试会提示选择调试配置，选 **Python 文件 / 当前文件** | 自动生成 `.vscode/launch.json`，程序开始运行 |
| 5 | 若提示 `debugpy` 缺失，在终端执行 `python3 -m pip install debugpy` 后重试 | 按 `F5` 能停在断点上 |

首次按 `F5` 自动生成的 `.vscode/launch.json` 大致如下（也可手工新建后粘贴）：

```json
{
  "version": "0.2.0",
  "configurations": [
    {
      "name": "Python: 当前文件",
      "type": "debugpy",
      "request": "launch",
      "program": "${file}",
      "console": "integratedTerminal"
    }
  ]
}
```

> **`"program": "${file}"`** 表示"调试当前打开的这个文件"，所以本节所有文件都可以用同一份配置，切换文件即切换调试目标。

---

# 第一部分　知识讲解

## 5A.1 场景导入：三种"看程序"的方式

程序结果不对时，初学者第一反应是"再读一遍代码"，第二反应是"多打几个 `print`"。三种方式的分工其实很清楚：

| 方式 | 能看到什么 | 代价 | 什么时候用 |
|---|---|---|---|
| 读代码 | 你**想到**的可能性 | 0 | 逻辑简单的笔误，如 `+` 写成 `-` |
| 加 `print` | 你**指定**的几个变量 | 要改源码、要删、输出刷屏 | 只想知道某一行前后某几个值 |
| **调试器** | **全部**变量、每一步的执行路径、调用层级 | 要学几个快捷键 | 结果错但看不出错在哪；循环里某一轮才出错 |

差别最典型的一幕：一个累加器被写在循环**内部**，每轮都被清零。坐在屏幕前读代码，很容易把 `total = 0` 看成"初始化"而放过；加 `print` 若打在循环**外**，你只会看到一个偏小的结果，看不出问题在哪一轮。而调试器只要在循环里按三次 `F10`，就会亲眼看到 `total` 从 `820` 变成 `0` 再变成 `1100`——**看见，就不用猜**。

本节的七个技能，就是把这套"看见"的能力装到手上。

## 5A.2 知识点 5A.1　断点与启动调试

### 5A.2.1 集中讲解

**断点（Breakpoint）** 是"让程序走到这一行先停住"的标记。调试的本质就是"在可疑的位置停一下，看清楚再走"。

**设断点的三种方式**：

1. 鼠标移到行号左侧的空白区，出现浅色圆点，单击即变**红色实心圆**——断点成功；
2. 光标停在目标行，按 `F9` 切换断点（再按一次取消）；
3. 右键行号 →「添加断点」。

**启动调试**：按 `F5`。注意与「直接运行」区分——直接运行（`Ctrl+F5` / 右上角三角按钮）**不会**在断点处停下，必须用 `F5` 或左侧「运行和调试」视图的绿色三角。

程序停在断点处时，当前行会**高亮为黄色**，表示"这一行还没执行"。顶部出现调试工具栏，从左到右依次是：

| 按钮 | 快捷键 | 含义 | 什么时候用 |
|---|---|---|---|
| 继续 | `F5` | 跑到下一个断点 | 这一段确认无误，跳到下一个检查点 |
| 逐过程 | `F10` | 执行本行，**不进**函数内部 | 只关心结果，不关心函数怎么算的 |
| 逐语句（步入） | `F11` | 执行本行，**进入**函数内部 | 怀疑函数里算错了 |
| 单步跳出 | `Shift+F11` | 把当前函数**执行完**并返回调用处 | 函数内部看够了 |
| 重启 | `Ctrl+Shift+F5` | 从头再来一遍 | 改了代码，想重跑 |
| 停止 | `Shift+F5` | 结束调试会话 | 已定位问题 |

**调试界面的五个区域**（本节后面反复用到）：

| 区域 | 位置 | 内容 |
|---|---|---|
| 编辑器 | 中间 | 黄色高亮行 = 即将执行的行；红点 = 断点 |
| 变量 | 左侧「运行和调试」视图 | 当前作用域的局部变量、全局变量，可逐层展开 |
| 监视 | 同上 | 自己写的表达式，如 `len(flows)`、`total / 3` |
| 调用堆栈 | 同上 | 谁调用了谁，点一下可跳到那一层 |
| 调试控制台 | 底部 | 调试中可直接输入表达式回车求值，如 `flows[0]` |

**四个易错点**：

- **断点是灰色空心圆**：说明断点"未生效"，常见原因是没选解释器、或调试的不是当前文件。按 `F5` 后若程序直接跑完没停，先查这一条。
- **按错键**：`Ctrl+F5` 是"运行"，`F5` 是"启动调试"，只差一个键，现象完全不同。
- **改了代码没重启**：调试会话用的是启动那一刻的代码，改完必须 `Ctrl+Shift+F5` 重启。
- **断点设太多**：一次设 5 个断点会把自己绕晕。**先设 1 个**，看完再挪。

**原理补充（概念深化）**：调试器并非另跑代码，而是 `debugpy` 通过 `sys.settrace` 在每行执行前插入钩子：解释器每要执行一行字节码先回调调试器，比对行号是否命中已注册断点，命中就挂起线程、把控制权交回 IDE。故黄色高亮行字节码尚未执行、变量面板里 `total` 还是旧值——这是'断点改变观察、不改变输出'的底层原因。`F5` 启动带 trace 的会话、`Ctrl+F5` 是裸解释器。断点设在循环体会被每轮重复命中；灰色空心圆说明未绑定解释器或文件，钩子比对不到自然不停。

### 5A.2.2 示例代码

**核心要点**：验证"断点 + `F5`"的基本动作——在累加行设一个断点，程序会在**每一轮循环**都停在同一行，按一次 `F5` 前进一轮，从而确认断点命中的次数等于循环次数。

In [ ]:
# ---------- 第 05A 节 · 示例 5A-1：断点与启动调试 ----------
# 对应知识点：5A.1 行断点（F9）与启动调试（F5）
# 操作：在 total = total + f 这一行的行号左侧单击，出现红点即断点成功；
#      按 F5 启动调试，程序会在断点处暂停（该行高亮为黄色），
#      再按 F5「继续」，会在下一轮循环再次停在同一个断点上。

flows = [820, 1100, 760]
total = 0
for f in flows:
    total = total + f          # ← 断点设在这一行
print("三断面合计：", total)
print("平均流量：", round(total / len(flows), 1))

**操作步骤**：

1. 打开 `示例5A1_断点与启动调试.py`，在 `total = total + f` 行号左侧单击，出现红点；
2. 按 `F5` 启动 → 程序停在该行，行号高亮为黄色；
3. 看左侧「变量」面板：`f = 820`、`total = 0`（这一行还没执行，所以 `total` 仍是 0）；
4. 按 `F5`（继续）→ 停在第 2 轮，`f = 1100`、`total = 820`；
5. 再按 `F5` 两次 → 一次停在第 3 轮，一次让程序跑完。

**运行结果**（直接运行，不设断点时）：

```text
三断面合计： 2680
平均流量： 893.3
```

> 断点不会改变程序的输出，它只改变"你看不看得到过程"。这是调试与改代码的根本区别：**调试是观察，不是修改**。

### 5A.2.3 练习代码

**任务**：输出把"最低"和"最高"报反了。用断点确认 `sort()` 之后 `speeds` 的真实顺序，然后只改一行把结果改对。

**操作清单**：

1. 在 `print("最低：", ...)` 这一行设断点，按 `F5`；
2. 记录暂停时 `speeds` 的完整内容（展开列表看每个元素）；
3. 按 3 次 `F10`，观察光标走了哪几行、哪些变量变了；
4. 回答：`speeds[-1]` 是列表里的**第几个**元素？它是最小值还是最大值？
5. 只改一行代码，使输出变为「最低： 45 最高： 78」。

In [ ]:
# ---------- 第 05A 节 · 练习 5A-1：断点与启动调试 ----------
# 任务：输出把"最低"和"最高"报反了，用断点确认排序后的真实顺序
# 操作清单：
#   1) 在 print("最低：", ...) 这一行设断点；
#   2) F5 启动，记录暂停时 speeds 的完整内容；
#   3) 分别按一次 F10，观察光标走了几行、变量有没有变化；
#   4) 只改一行代码把结果改对。

speeds = [62, 45, 78, 51]
speeds.sort()
print("升序车速：", speeds)
print("最低：", speeds[-1], "最高：", speeds[0])     # ← 断点设在这一行
avg = sum(speeds) / len(speeds)
print("平均车速：", round(avg, 1))

**运行结果**（未修复）：

```text
升序车速： [45, 51, 62, 78]
最低： 78 最高： 45
平均车速： 59.0
```

**自检要点**：

- [ ] 能说出程序共命中断点 **1 次**（不在循环里）；
- [ ] 能说出 `speeds[-1]` 是升序列表的**最后一个**，也就是最大值；
- [ ] 修复后输出「最低： 45 最高： 78」，平均车速仍为 59.0。

**变式代码（迁移练习）**：把『每轮暂停时变量面板显示的值』直接打印成对照表，验证断点命中次数等于循环次数、且暂停行尚未执行；这是 Notebook 内可运行的对照代码，断点/单步等实际操作仍须在 Trae 中完成。

In [ ]:
# 对照：模拟在累加行设断点后，每轮暂停时变量面板应显示的值（该行尚未执行）
flows = [820, 1100, 760]
total = 0
hit = 0
for f in flows:
    hit = hit + 1
    print("断点命中第", hit, "次 | f =", f, "| total =", total, "（本行未执行）")
    total = total + f
print("循环结束：共命中", hit, "次 | 合计 =", total, "| 平均 =", round(total / len(flows), 1))

**预期输出**：

```text
断点命中第 1 次 | f = 820 | total = 0 （本行未执行）
断点命中第 2 次 | f = 1100 | total = 820 （本行未执行）
断点命中第 3 次 | f = 760 | total = 1920 （本行未执行）
循环结束：共命中 3 次 | 合计 = 2680 | 平均 = 893.3
```

### 5A.2.4 实操步骤（上机跟随）

1. 运行对照代码，核对共打印 3 行『断点命中第 n 次』，与 flows 的 3 轮循环一一对应。
2. 看第 1 行 total = 0，确认断点停住的累加行尚未执行、变量面板显示的是旧值。
3. 看第 2 行 total = 820、第 3 行 total = 1920，确认每次继续后 total 才更新为上一轮之和。
4. 看末行『共命中 3 次 | 合计 = 2680 | 平均 = 893.3』，确认断点不改变程序输出。
5. 在 Trae 中把断点设在 total = total + f 行、按 F5 启动，核对命中次数与这些值一致。

### 5A.2.5 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 按了 Ctrl+F5 断点完全不停 | Ctrl+F5 是直接运行，不启动调试会话 | 按 F5 启动调试 |
| 断点是灰色空心圆 | 未绑定解释器或调试的不是当前文件 | 先 Python: Select Interpreter，确认 F5 调试当前文件 |
| 改了代码后断点行为没变 | 调试会话用的是启动那一刻的代码 | 按 Ctrl+Shift+F5 重启 |
| 暂停行明明高亮，值却没更新 | 黄色行是即将执行、字节码尚未运行 | 再按一次 F10 才看到该行结果 |
| 一次设 5 个断点把自己绕晕 | 断点过多、观察焦点分散 | 先设 1 个，看完再挪 |

## 5A.3 知识点 5A.2　逐过程（`F10`）

### 5A.3.1 集中讲解

**逐过程**（Step Over，也叫"单步跳过"）的规则只有一句：**执行当前这一行，然后停在下一行；如果这一行里调用了函数，不进去，直接把函数当作一步走完。**

它的价值在于"跟着数据的脚步走"。循环 + 累加是逐过程最好的舞台：每按一次 `F10`，你就看到一次"变量被改成了什么"。

| 场景 | 用 `F10` 的结果 |
|---|---|
| 停在 `total = total + f` | 执行加法，`total` 立刻更新为新值 |
| 停在 `print(...)` | 先在下方输出，再停在下一行 |
| 停在 `g = phase_green(...)` | `phase_green` 整个跑完，`g` 直接拿到返回值 |
| 停在 `for f in flows:` | 取下一个元素（或结束循环，跳到循环之后） |

**读循环时的三个观察点**（这是本节最实用的技巧）：

1. **进入循环前**：循环变量的初值、累加器的初值是多少？
2. **每一轮结束**：累加器**只增不减**吗？如果发现它"变小了"或"回到初值"，问题就在这一轮里。
3. **循环结束后**：`for` 的变量最后停在最后一个元素，而累加器应是全部之和——两者对比即可判断有没有漏加、多加。

**易错点**：

- `F10` 一次只走**一行**，但一行里可能有多个操作（如 `a = b + c * d`），想细看就得把它拆成几行——**能拆开的代码才好调试**，这也是"一行只做一件事"的理由。
- 停在 `for` 行按 `F10`，是"取下一个元素并跳到循环体第一行"，不会执行循环体。
- 在 `F10` 途中按 `F5`，会直接跳到下一个断点（若没有下一个断点，程序跑完）。

**原理补充（概念深化）**：`F10` 与 `F11` 共用行级 trace 钩子，区别只在遇到 `CALL` 字节码时：`F10` 让被调函数在当前步整个跑完、直接拿返回值不进函数体；`F11` 把暂停点下移到被调函数第一行。因此 `F10` 一次前进一个源码行，一行含多个运算时挂起粒度仍是一行，想看中间值只能拆行——'一行只做一件事'本质是让观察粒度更细。累加器每按一次就看到新值，若变小或回初值说明该行重赋了初值。停在 `for` 行按 `F10` 只取下一元素、不执行循环体。

### 5A.3.2 示例代码

**核心要点**：验证 `F10` 与"变量随时间变化"的对应关系——在累加行设断点后，每按一次 `F10`，`total` 依次变成 820 → 1920 → 2680，三次相加一步不差。

In [ ]:
# ---------- 第 05A 节 · 示例 5A-2：逐过程（F10）观察累加 ----------
# 对应知识点：5A.2 逐过程 F10 —— 一行一行执行，不进入函数内部
# 操作：在 total = total + f 设断点 → F5 启动 → 连按 3 次 F10，
#      每次停顿都看一眼「变量」面板里的 total，把数值记下来。

flows = [820, 1100, 760]
total = 0
for f in flows:
    total = total + f          # ← 断点设在这一行，按 F10 走完这一行
    print("本轮 f =", f, "累计 total =", total)
print("总流量：", total)
print("平均流量：", round(total / len(flows), 1))

**操作步骤**：

1. 在 `total = total + f` 设断点，按 `F5`；
2. 连按 3 次 `F10`，每次停顿都在「变量」面板记下 `f` 与 `total`；
3. 填表：

| 第几次 `F10` | `f` | `total` |
|---|---|---|
| 1 | 820 | 820 |
| 2 | 1100 | 1920 |
| 3 | 760 | 2680 |

4. 连续按 `F10`，直到光标停在 `for` 之后的第一条 `print` 行，记录实际按键次数；再按 1 次 `F10` 走完最后一条 `print`，核对两条 `print` 的输出与表格中的最终值一致。

**运行结果**：

```text
本轮 f = 820 累计 total = 820
本轮 f = 1100 累计 total = 1920
本轮 f = 760 累计 total = 2680
总流量： 2680
平均流量： 893.3
```

### 5A.3.3 练习代码

**任务**：三个断面 `820 + 1100 + 760` 应得 2680，程序却输出 760。用 `F10` 盯住 `total` 的取值序列，说出它在**哪一行**被打回原形。

**操作清单**：

1. 在 `total = 0` 设断点，按 `F5`；
2. 连按 8 次 `F10`，把每一行的运行结果记下来；
3. 回答：`total` 的取值序列是什么？它在哪一行被改回 0？
4. 改一行代码（不要删掉初始化语句，只是挪个位置）使总流量变为 2680。

In [ ]:
# ---------- 第 05A 节 · 练习 5A-2：逐过程（F10）找累加错误 ----------
# 现象：三个断面 820 + 1100 + 760 应得 2680，程序却只有 760，平均流量 253.3
# 任务：用 F10 逐行执行，在循环里盯住 total 的变化规律，说出它在第几行被"打回原形"
# 操作：在 total = 0 设断点 → F5 → 反复按 F10，注意 total 的取值序列

flows = [820, 1100, 760]
total = 0
for f in flows:
    total = 0
    total = total + f
    print("本轮 f =", f, "累计 total =", total)
print("总流量：", total)
print("平均流量：", round(total / len(flows), 1))

**运行结果**（未修复）：

```text
本轮 f = 820 累计 total = 820
本轮 f = 1100 累计 total = 1100
本轮 f = 760 累计 total = 760
总流量： 760
平均流量： 253.3
```

**自检要点**：

- [ ] 能指出 `total` 的序列为 0 → 820 → 0 → 1100 → 0 → 760；
- [ ] 能说出"初始化必须在循环**外**"这一条规律；
- [ ] 修复后输出「总流量： 2680」与「平均流量： 893.3」。

**变式代码（迁移练习）**：把示例的三断面累加换成四段车速，并把每次 F10 后的 total 依次打印，验证累加器只增不减、末值等于 sum；这是 Notebook 内可运行的对照代码，断点/单步等实际操作仍须在 Trae 中完成。

In [ ]:
# 对照：把每次 F10 后 total 的值依次记录，验证累加器"只增不减、末值等于总和"
speeds = [62, 45, 78, 51]
total = 0
step = 0
for s in speeds:
    total = total + s
    step = step + 1
    print("第", step, "次 F10：s =", s, "→ total =", total)
print("末值：", total, "| 等于 sum：", total == sum(speeds), "| 平均：", round(total / len(speeds), 1))

**预期输出**：

```text
第 1 次 F10：s = 62 → total = 62
第 2 次 F10：s = 45 → total = 107
第 3 次 F10：s = 78 → total = 185
第 4 次 F10：s = 51 → total = 236
末值： 236 | 等于 sum： True | 平均： 59.0
```

### 5A.3.4 实操步骤（上机跟随）

1. 运行对照代码，核对依次打印 4 行『第 n 次 F10』，total 从 62 递增到 236。
2. 看每行走完『total = total + s』后的值：62 → 107 → 185 → 236，确认只增不减。
3. 看末行『等于 sum：True』，确认循环结束累计值等于 sum(speeds)。
4. 看『平均：59.0』，确认 236 / 4 的统计结果与逐过程看到的一致。
5. 在 Trae 中于累加行设断点、连按 F10，把变量面板里的 total 与这 4 个值逐一核对。

### 5A.3.5 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| F10 一次跳过了整个函数 | Step Over 把被调函数当作一步跑完 | 怀疑函数内部改用 F11 步入 |
| 一行里多个操作看不出中间值 | F10 粒度是源码行，不是单个运算 | 把 a = b + c*d 拆成多行再调 |
| 停在 for 行按 F10 直接跳出了循环 | 此时是取下一元素或结束循环，不执行循环体 | 把观察点放到循环体内的行 |
| F10 途中程序一下跑完了 | 途中按了 F5 会跳到下一个断点 | 确认没有其它断点，或逐步重来 |
| 累加器某轮突然变小 | 该行重新赋了初值，初始化误写在循环内 | 把初始化移到循环外，再按 F10 复核 |

## 5A.4 知识点 5A.3　步入（`F11`）

### 5A.4.1 集中讲解

**步入**（Step Into）的规则：**执行当前这一行；如果这一行调用了函数，就进到那个函数的第一行去。**

什么时候需要进去？只有一个判断标准：**结果不对，且错误可能在函数内部**。典型信号是"函数名和注释都说它该返回什么，但调用处拿到的值就是不对"。

进入函数后有两件事必看：

1. **形参的值**——传进来的实参，和你想的一样吗？参数顺序反了、类型不对，都会在这里露馅；
2. **函数内的局部变量**——它们只在函数内部存在，用 `print` 根本打不到（函数一结束就没了），这恰恰是调试器的主场。

| 操作 | 结果 |
|---|---|
| 停在调用行按 `F11` | 进入函数体第一行（`def` 下一行） |
| 停在 `print(avg_speed(data))` 按 `F11` | 先进入 `avg_speed`，返回后才执行 `print` |
| 在函数最后一行按 `F10` | 相当于"返回"，光标回到调用处 |

**易错点**：

- **步入到标准库**：停在 `print(...)` 或 `len(...)` 按 `F11`，会跳进 Python 自带的库源码（一堆看不懂的英文），这不是出错。按 `Shift+F11` 跳出来即可，之后遇到内置函数改用 `F10`。
- **步入没有函数调用的行**：光标只是前进一行，和 `F10` 效果相同——所以 `F11` 可以放心连按。
- **以为"传参"是安全的**：参数顺序错（`f(a, b)` 写成 `f(b, a)`）不会报错，只会算错，而且两个参数类型相同时最难发现（`5` 分钟和 `12` 辆车都是整数）。

**原理补充（概念深化）**：函数调用会压入新栈帧（frame），保存本次形参、局部变量与返回地址；`F11` 让调试器在新帧第一行挂起，变量面板切成该帧局部作用域——`print` 打不到的函数内变量此刻才可见。形参只是帧里的名字、按位置与实参绑定：解释器不做类型语义检查，只把第 1 个实参给第 1 个形参，两个参数同为 int 时顺序写反既不报错也难察觉、只会算错，这正是'靠步入看形参'的价值。停在 `print()` 按 `F11` 会钻进标准库，按 `Shift+F11` 弹回即可。

### 5A.4.2 示例代码

**核心要点**：验证 `F11` 能"钻进"函数体——停在调用行按 `F11`，光标跳到 `def classify(speed)` 内部，此时「变量」面板立刻出现形参 `speed`，其值就是调用处传进去的那个车速。

In [ ]:
# ---------- 第 05A 节 · 示例 5A-3：步入（F11）进入函数内部 ----------
# 对应知识点：5A.3 步入 F11 —— 光标停在函数调用处，按 F11 进入函数体
# 操作：在 print(s, "→", classify(s)) 设断点 → F5 → 按 F11，
#      光标跳到 def classify(speed) 内部，此时「变量」面板能看到形参 speed 的值。

def classify(speed):
    """按车速判定等级（km/h）"""
    if speed >= 80:
        return "超速"
    if speed >= 60:
        return "正常"
    return "偏慢"

speeds = [92, 58, 74]
for s in speeds:
    print(s, "→", classify(s))     # ← 断点设在这一行，然后按 F11

**操作步骤**：

1. 在 `print(s, "→", classify(s))` 设断点，按 `F5`；
2. 按 `F11` → 光标进入 `classify`，看 `speed = 92`；
3. 连按 `F10` 走完 `if` 分支，看它从哪一行 `return`；
4. 最后一次 `F10` 回到调用行，此时 `classify(s)` 已有结果；
5. 再按 `F5`，检查第 2、3 轮是否分别传入 58、74。

**运行结果**：

```text
92 → 超速
58 → 偏慢
74 → 正常
```

### 5A.4.3 练习代码

**任务**：5 分钟观测到 12 辆车，折算成小时流量只有 25 辆（应为 144）。在 `hourly_flow(...)` 调用处按 `F11` 进入函数，比较**形参**与**实参**的对应关系，找出传参错误。

**操作清单**：

1. 在 `flow = hourly_flow(minutes, vehicles)` 设断点，按 `F5`；
2. 记下调用行里 `vehicles`、`minutes` 的值（12 和 5）；
3. 按 `F11` 进入函数，记下形参 `vehicles`、`minutes` 的值；
4. 对比两组值，说出"谁被当成了谁"；
5. 改调用处的一行，使三行输出变为 144 / 120 / 200。

In [ ]:
# ---------- 第 05A 节 · 练习 5A-3：步入（F11）检查形参 ----------
# 现象：5 分钟观测 12 辆车，折成小时流量只有 25 辆，明显偏小（应为 144）
# 任务：在 hourly_flow(...) 调用处按 F11 进入函数，比较「形参」与「实参」的对应关系
# 提示：进入函数后先看 vehicles 与 minutes 各是多少，再回到调用处看传的是什么

def hourly_flow(vehicles, minutes):
    """由 minutes 分钟的观测值折算为小时流量（辆/小时）"""
    return round(vehicles / minutes * 60)

samples = [(12, 5), (8, 4), (20, 6)]
for vehicles, minutes in samples:
    flow = hourly_flow(minutes, vehicles)      # ← 断点设在这一行，按 F11
    print("5 分钟内", vehicles, "辆 → 小时流量", flow, "辆/小时")

**运行结果**（未修复）：

```text
5 分钟内 12 辆 → 小时流量 25 辆/小时
5 分钟内 8 辆 → 小时流量 30 辆/小时
5 分钟内 20 辆 → 小时流量 18 辆/小时
```

**自检要点**：

- [ ] 进入函数后能看到 `vehicles = 5`、`minutes = 12`（与调用处的 12、5 正好相反）；
- [ ] 能说清"实参顺序与形参顺序必须一一对应"；
- [ ] 修复后三行分别为 144、120、200 辆/小时。

**变式代码（迁移练习）**：定义 hour_flow 并用两组样本分别打印调用处实参与函数内形参，验证按位置一一对应；这是 Notebook 内可运行的对照代码，断点/单步等实际操作仍须在 Trae 中完成。

In [ ]:
# 对照：步入函数时变量面板显示的形参与调用处实参，验证传参顺序一一对应
def hourly_flow(vehicles, minutes):
    print("  [步入] 形参 vehicles =", vehicles, "minutes =", minutes)
    return round(vehicles / minutes * 60)
samples = [(12, 5), (8, 4)]
for vehicles, minutes in samples:
    print("[调用处] 实参 vehicles =", vehicles, "minutes =", minutes)
    flow = hourly_flow(vehicles, minutes)
    print("  → 小时流量 =", flow, "辆/小时")

**预期输出**：

```text
[调用处] 实参 vehicles = 12 minutes = 5
  [步入] 形参 vehicles = 12 minutes = 5
  → 小时流量 = 144 辆/小时
[调用处] 实参 vehicles = 8 minutes = 4
  [步入] 形参 vehicles = 8 minutes = 4
  → 小时流量 = 120 辆/小时
```

### 5A.4.4 实操步骤（上机跟随）

1. 运行对照代码，每组样本先打印『[调用处] 实参』，再打印『[步入] 形参』，最后打印小时流量。
2. 核对第 1 组调用处 vehicles = 12、minutes = 5，与函数内形参 12、5 一一对应，结果 144。
3. 核对第 2 组 8、4 对应 8、4，结果 120，确认按位置绑定、顺序一致。
4. 把调用改成 hourly_flow(minutes, vehicles) 再跑，形参变成 5、12，结果错误却不报错。
5. 在 Trae 中于调用行按 F11 进入函数体，核对变量面板里形参与调用处实参是否一一对应。

### 5A.4.5 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 按 F11 跳进了看不懂的英文源码 | 步进了 print / len 等标准库函数 | 按 Shift+F11 跳出，之后遇内置函数改用 F10 |
| 两个参数都是整数，传反了却没报错 | Python 按位置绑定、不做类型语义检查 | 靠 F11 看形参值，逐一核对顺序 |
| 函数内局部变量 print 打不到 | 局部变量随函数结束即销毁 | 用调试器在函数内暂停查看，而不是加 print |
| 按 F11 光标只前进一行 | 该行没有函数调用，F11 与 F10 等效 | 无需担心，可放心连按 |
| 形参值与预期不一致 | 实参顺序或个数写错 | 回到调用处核对实参排列 |

## 5A.5 知识点 5A.4　步出（`Shift+F11`）

### 5A.5.1 集中讲解

**步出**（Step Out）的规则：**把当前函数的剩余部分一次性执行完，然后停在调用它的那一行**（此时函数的返回值已经拿到）。

它解决两个具体问题：

1. **函数看够了**：内部逻辑已经确认正确，没必要一行行走完，直接跳回调用处继续；
2. **确认返回值**：最关键的用法——跳出后立刻看调用处的变量，**它是不是你期望的值**。如果函数里算得对、外面拿到的却不对，问题往往在 `return` 上（漏写 `return`、`return` 写在了 `if` 分支里导致某些情况走不到）。

| 现象 | 跳出后看到什么 | 结论 |
|---|---|---|
| 函数内 `print` 显示 2680，跳出后 `total` 是 `None` | 拿到 `None` | 函数**没有 `return`**（只打印不交回结果） |
| 函数内 `print` 显示 2680，跳出后 `total` 是 2680 | 一致 | 函数没问题，错在调用之后 |
| 跳出后 `total` 是个"看着对但差一点"的数 | 有值但不对 | 函数内部算式有错，需按 `F11` 再进去 |

**易错点**：

- `Shift+F11` 是"执行完当前函数"，若当前层是**最外层模块**（没有函数包着），它可能直接结束整个调试会话——按之前先看光标停在哪个函数里。
- 跳出**不经过**函数内剩余行上的断点（准确说：会执行，但不停）。想在函数内停就别急着跳出。
- 跳出后光标停在"调用行的下一行"或"调用行本身"取决于语言与调试器版本，**看变量值比看光标位置可靠**。

**原理补充（概念深化）**：`Shift+F11` 给当前帧设一个运行到返回的临时断点，把该帧剩余字节码一次执行完、在返回指令后弹栈停到调用处，跳过的是整个当前函数而非某一行。它最值钱的是核对返回值：函数内算对、跳出后却拿到 `None`，几乎可断定漏了 `return`，因为函数体执行到末尾都没有返回语句时，解释器会隐式补一条返回空值，只打印不交回的值自然到不了调用栈。若跳出后的值看着对却差一点，说明算式有误、须再步入。注意光标已在最外层模块时步出等于结束会话。

### 5A.5.2 示例代码

**核心要点**：验证 `Shift+F11` 的"带着返回值回家"——在调用行设断点，`F11` 进入函数后直接 `Shift+F11` 跳出，调用行的 `result` 立刻有了值，且与函数内算出的结果一致。

In [ ]:
# ---------- 第 05A 节 · 示例 5A-4：步出（Shift+F11）看返回值 ----------
# 对应知识点：5A.4 步出 Shift+F11 —— 函数内部看够了，一次跳出并把返回值带回调用处
# 操作：在 result = avg_speed(data) 设断点 → F5 → F11 进入函数 →
#      按 Shift+F11 直接跳出，光标回到调用行，result 立刻有了值。

def avg_speed(speeds):
    """平均车速，保留 1 位小数"""
    total = 0
    for s in speeds:
        total = total + s
    return round(total / len(speeds), 1)

data = [62, 45, 78, 51]
result = avg_speed(data)       # ← 断点设在这一行，F11 进入、Shift+F11 跳出
print("平均车速：", result, "km/h")

**操作步骤**：

1. 在 `result = avg_speed(data)` 设断点，按 `F5`；
2. 按 `F11` 进入 `avg_speed`，看 `total` 的累加；
3. 连按两次 `F10` 走到 `return` 行之前，记下此刻 `total = 236`；
4. 按 `Shift+F11` 跳出 → 光标回到调用行，`result = 59.0`；
5. 按 `F10` 走完 `print`，核对输出。

**运行结果**：

```text
平均车速： 59.0 km/h
```

> `236 / 4 = 59.0`。函数内的 `total` 与调用处的 `result` 完全对得上，说明这个函数是"可信"的——以后调它就不用再进来了。

### 5A.5.3 练习代码

**任务**：函数内部明明打印了 2680，函数外面拿到的 `total` 却是 `None`。在调用行设断点，`F11` 进去、`Shift+F11` 跳出，观察跳出瞬间 `total` 的值，判断"函数有没有把结果交回来"。

**操作清单**：

1. 在 `total = total_flow(flows)` 设断点，按 `F5`；
2. 按 `F11` 进入函数，按 `F10` 走到最后一行（此时内部 `print` 已输出 2680）；
3. 按 `Shift+F11` 跳出，立刻看 `total` 的值；
4. 补上一行使 `total` 变成 2680。

In [ ]:
# ---------- 第 05A 节 · 练习 5A-4：步出（Shift+F11）检查返回值 ----------
# 现象：函数内部明明打印了 2680，函数外面拿到的 total 却是 None
# 任务：在 total = total_flow(flows) 设断点，F11 进入、Shift+F11 跳出，
#      观察跳出瞬间 total 的值，从而判断函数"有没有把结果交回来"
# 提示：函数内部算对了却交不出结果，问题通常不在算，而在最后一行

def total_flow(flows):
    """合计流量"""
    total = 0
    for f in flows:
        total = total + f
    print("函数内部合计：", total)        # 内部算对了
    # 这里少了一行把结果交回去的语句

flows = [820, 1100, 760]
total = total_flow(flows)      # ← 断点设在这一行，F11 进入、Shift+F11 跳出
print("总流量：", total, "辆")

**运行结果**（未修复）：

```text
函数内部合计： 2680
总流量： None 辆
```

**自检要点**：

- [ ] 能说出跳出后 `total` 是 `None`（不是 0、不是 2680）；
- [ ] 能解释"只 `print` 不 `return`，函数对外返回 `None`"；
- [ ] 修复后输出「总流量： 2680 辆」。

**变式代码（迁移练习）**：并列定义有 return 与只 print 的两个函数，打印步出后调用处分别拿到 2680 与 None；这是 Notebook 内可运行的对照代码，断点/单步等实际操作仍须在 Trae 中完成。

In [ ]:
# 对照：步出后调用处拿到的返回值——有 return 得结果，只 print 不 return 得 None
def with_return(flows):
    return sum(flows)
def no_return(flows):
    print("  [函数内] 合计 =", sum(flows))
data = [820, 1100, 760]
r1 = with_return(data)
r2 = no_return(data)
print("有 return 的调用处：", r1)
print("无 return 的调用处：", r2)

**预期输出**：

```text
  [函数内] 合计 = 2680
有 return 的调用处： 2680
无 return 的调用处： None
```

### 5A.5.4 实操步骤（上机跟随）

1. 运行对照代码，注意 no_return 内部先打印『[函数内] 合计 = 2680』，但外部拿到的却是 None。
2. 看『有 return 的调用处：2680』，确认 return 把结果交回了调用处。
3. 看『无 return 的调用处：None』，确认只 print 不 return 时外部得到 None。
4. 给 no_return 补一行 return sum(flows) 再跑，外部应变为 2680。
5. 在 Trae 中于调用行 F11 进入、再 Shift+F11 跳出，核对 r1 / r2 与这两行输出一致。

### 5A.5.5 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 函数里算对了，外面却是 None | 函数体执行到末尾没有 return，隐式返回 None | 补上 return 结果 |
| Shift+F11 直接结束了整个调试 | 当前已在最外层 module，弹掉最后一帧即结束会话 | 按之前先看光标停在哪个函数里 |
| 跳出后函数内剩余行的断点没停 | 步出会执行完当前函数剩余部分但不停在其断点上 | 想在函数内停就别急着步出 |
| 盯着光标位置判断返回值不可靠 | 跳出后停在哪行取决于调试器版本 | 以调用处变量的实际值为准 |
| 跳出后值看着对但差一点 | 函数内部算式有误 | 按 F11 再进去逐行复核 |

## 5A.6 知识点 5A.5　查看变量与方法

### 5A.6.1 集中讲解

停下来之后，"看"的手段有四种，各有分工：

| 手段 | 怎么操作 | 适合看什么 | 优点 |
|---|---|---|---|
| 变量面板 | 左侧「变量」区，点箭头展开 | 当前作用域**全部**变量，含列表 / 字典 / 元组 | 不用想，全都列出来 |
| 鼠标悬停 | 光标停在代码里的变量上 | 单个变量的值、字符串长度、列表元素数 | 最快，1 秒 |
| 监视（Watch） | 「监视」区点 `+`，输入表达式 | `len(flows)`、`total / 3`、`flows[0]` 等**计算结果** | 一次写好，全程跟着刷新 |
| 调试控制台 | 底部输入框，回车求值 | 临时算一句，如 `sorted(speeds)`、`sum(road_flows["京开"])` | 不改代码就能试 |

**展开嵌套结构的技巧**：字典 / 列表在面板里默认是折叠的，左边有三角箭头。看"字典里每个键对应的列表有多长"，比看"字典长什么样"有用得多——案例 5A-2 的 bug 1 就是靠这一步发现的。

**"方法"怎么看**：变量面板里对象展开后能看到它的属性和方法（如列表的 `append`、字符串的 `strip`），但**方法本身是"动作"不是"值"**，看方法要看**调用结果**：把 `flows.count(820)`、`plate[:2]` 这类表达式写进「监视」，得到的就是方法执行后的值。这是初学者最容易忽略的一点：**监视里可以写调用，不只是写名字**。

**易错点**：

- **面板里没有某个变量**：说明还没执行到定义它的那一行（黄色高亮行还没跑），按一次 `F10` 就出现了。
- **监视表达式报错**（显示 `not defined`）：表达式里用了当前作用域没有的名字，或还没执行到那一行。
- **只看最外层**：字典折叠时只显示 `{…}`，看不出每个键背后装了几条记录；展开才发现 `'京港澳': [39]` 只有 1 个元素——本节的 bug 就藏在这种地方。
- **在监视里写 `print(...)`**：监视只求值，不输出；要输出用调试控制台。

**原理补充（概念深化）**：调试器挂起后能通过帧对象 `f_locals` 枚举当前作用域的全部变量，这是变量面板无需挑选、全部列出的来源；字典与列表默认折叠，只因它们是引用，要展开才顺着引用看到嵌套元素，某个键背后只有一条记录这种问题，不展开根本发现不了。监视与悬停共用表达式求值器，不仅能写变量名，还能写 `len(flows)` 这类调用，调试器在每次挂起时都重新求值并刷新，所以会随断点位置与循环轮次自动变化。字典长度与列表长度含义不同，并列写进监视最能戳破看错容器的错误。监视只求值不输出，未定义的名字会显示未定义。

### 5A.6.2 示例代码

**核心要点**：验证四种"看"的手段在用同一个断点时各能看到什么——变量面板看层级、悬停看单值、监视看表达式、调试控制台临时试算，四者看到的 `len(flows)`、`sum(flows)`、`max(flows)` 必须一致。

In [ ]:
# ---------- 第 05A 节 · 示例 5A-5：变量面板、悬停与监视表达式 ----------
# 对应知识点：5A.5 查看变量与方法 —— 变量面板展开、鼠标悬停、监视 Watch
# 操作：
#   1) 在 print(...) 设断点 → F5 暂停；
#   2) 「变量」面板展开 road_flows，看字典 → 列表的层级；
#   3) 鼠标悬停在 len(flows) 上，直接看到结果；
#   4) 在「监视」里添加表达式：len(flows)、sum(flows)、max(flows)。

road_flows = {"京港澳": [820, 1100, 760], "京开": [640, 720]}
for road in road_flows:
    flows = road_flows[road]
    print(road, "断面数：", len(flows), "| 合计：", sum(flows), "| 峰值：", max(flows))

**操作步骤**：

1. 在 `print(...)` 设断点，按 `F5`；
2. 「变量」面板展开 `road_flows` → 展开 `京港澳`，数一数列表里几个元素；
3. 鼠标悬停在 `len(flows)` 上，直接读到 `3`；
4. 「监视」里添加三条：`len(flows)`、`sum(flows)`、`max(flows)`；
5. 在调试控制台输入 `road_flows["京开"]` 回车，看第二个断面的数据；
6. 按 `F5` 到第二轮，观察三条监视表达式的值**自动刷新**为 `2 / 1360 / 720`。

**运行结果**：

```text
京港澳 断面数： 3 | 合计： 2680 | 峰值： 1100
京开 断面数： 2 | 合计： 1360 | 峰值： 720
```

### 5A.6.3 练习代码

**任务**：京港澳有 3 个断面，输出却写「断面数：2」，而京开只有 2 个，恰好也显示 2。在 `print(...)` 设断点，把 `len(road_flows)` 与 `len(flows)` **同时**加入监视，看两个值的差别。

**操作清单**：

1. 在 `print(...)` 设断点，按 `F5`；
2. 监视里加 `len(road_flows)` 和 `len(flows)`，记下两者的值；
3. 按 `F5` 到第二轮，再看一次——哪个值变了、哪个没变？
4. 回答：代码里写的是哪个容器的长度？应该写哪个？
5. 改一处，使输出变为「断面数： 3」与「断面数： 2」。

In [ ]:
# ---------- 第 05A 节 · 练习 5A-5：变量的"范围"看错了 ----------
# 现象：京港澳有 3 个断面，输出却写"断面数：2"；京开只有 2 个，恰好也显示 2
# 任务：在 print(...) 设断点，把 len(road_flows) 与 len(flows) 同时加入「监视」，
#      看两个值的差别，说清"2"是哪个容器的长度
# 提示：监视表达式里可以写任何当前作用域内合法的表达式，不必改动源代码

road_flows = {"京港澳": [820, 1100, 760], "京开": [640, 720]}
for road in road_flows:
    flows = road_flows[road]
    print(road, "断面数：", len(road_flows), "| 合计：", sum(flows))   # ← 断点设在这一行

**运行结果**（未修复）：

```text
京港澳 断面数： 2 | 合计： 2680
京开 断面数： 2 | 合计： 1360
```

**自检要点**：

- [ ] 能让两条监视同时显示，并说出 `len(road_flows)` 恒为 2 的原因（字典只有 2 个键）；
- [ ] 能说出"监视里的表达式会随断点位置自动刷新"；
- [ ] 修复后两行分别为 3 与 2。

**变式代码（迁移练习）**：把监视里并列写 len(字典) 与 len(列表) 的结果直接打印，验证两者不是一回事；这是 Notebook 内可运行的对照代码，断点/单步等实际操作仍须在 Trae 中完成。

In [ ]:
# 对照：模拟「监视」里并列写 len(字典) 与 len(列表)，看清两者不是一回事
road_flows = {"京港澳": [820, 1100, 760], "京开": [640, 720]}
print("监视 len(road_flows) =", len(road_flows), "（字典键数，恒定）")
for road in sorted(road_flows):
    flows = road_flows[road]
    print("监视", road, "：len(flows) =", len(flows),
          "| sum =", sum(flows), "| max =", max(flows))
print("调试控制台试算 sorted(road_flows) =", sorted(road_flows))

**预期输出**：

```text
监视 len(road_flows) = 2 （字典键数，恒定）
监视 京开 ：len(flows) = 2 | sum = 1360 | max = 720
监视 京港澳 ：len(flows) = 3 | sum = 2680 | max = 1100
调试控制台试算 sorted(road_flows) = ['京开', '京港澳']
```

### 5A.6.4 实操步骤（上机跟随）

1. 运行对照代码，先看『监视 len(road_flows) = 2』，确认这是字典的键数、恒定不变。
2. 看京开一行 len(flows) = 2、sum = 1360、max = 720，京港澳一行 len = 3、sum = 2680、max = 1100。
3. 对比两个 len：字典恒为 2，而各断面的 flows 分别是 2 与 3，确认『看错了容器』。
4. 看末行 sorted(road_flows) 的固定顺序输出，确认字典遍历需排序才稳定。
5. 在 Trae 中于 print 行暂停，把 len(road_flows) 与 len(flows) 同时加进监视，展开字典里的列表核对元素个数。

### 5A.6.5 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 变量面板里找不到某个变量 | 还没执行到定义它的那一行（黄色行未跑） | 按一次 F10 再展开 |
| 监视表达式显示 not defined | 用了当前作用域没有、或还没执行到的名字 | 确认作用域与执行位置 |
| 只看到字典折叠的 {…} | 嵌套结构默认折叠，看不到每个键背后几条记录 | 点三角箭头逐层展开列表/字典 |
| 监视里写 print(...) 没有输出 | 监视只求值、不输出 | 要输出用调试控制台 |
| len(dict) 与 len(list) 混用 | 两者分别返回键个数与元素个数 | 并列写进监视，明确要看哪个容器 |

## 5A.7 知识点 5A.6　调用堆栈

### 5A.7.1 集中讲解

**调用堆栈**（Call Stack）回答一个问题：**"现在停的这一行，是被谁、经过哪几步调过来的？"**

面板里自上而下排列，**最上面是当前正在执行的函数**，往下依次是调用它的函数，最底下是程序入口（`<module>`，即"文件最外层"）。点任意一帧，编辑器会跳到那层的代码行，变量面板也切换成**那一层**的局部变量——这是它最值钱的能力：**同时看几层的现场**。

典型用法（案例 5A-2 的定位路径）：

| 步骤 | 操作 | 看到什么 |
|---|---|---|
| 1 | 在函数内部深处设断点 | 停在 `lane_density` 里 |
| 2 | 看堆栈：`judge` ← `report` ← `<module>` | 知道是 `report` 调的 `judge` |
| 3 | 点 `judge` 那一帧 | 看到 `judge` 里 `lanes = 3` |
| 4 | 回到 `lane_density` 帧 | 看到这里的 `lanes` 是 `2` |
| 5 | 结论 | 错误在 `judge` **传参**那一行，不在算式 |

**三项注意**：

- 堆栈是**只读**的现场记录，不能"改历史"，只能看。
- 递归（第 6 章）时同一个函数会出现很多帧，从下往上数能看清"第几层递归"。
- 语法错误不会产生堆栈；堆栈只在**运行到某处暂停**（断点或异常）时才有内容。

**易错点**：

- 只看最上面一帧就下结论——错误常常在**上一层**的传参。
- 堆栈里出现了看不懂的库函数帧（如 `runpy`、`ipykernel`），那是解释器自己的调用，忽略即可。

**原理补充（概念深化）**：调用堆栈就是解释器里被函数调用逐层压入的一串帧：栈顶是当前正在执行的函数，往下依次是调用它的函数，最底下是模块最外层。调试器把它做成只读面板，点任意一帧就把编辑器与变量面板切到那一层的局部变量，于是一次暂停能同时看到上下几层的现场，不必靠打印把值层层传出来。形参写在各自的帧里，所以 `judge` 帧的车道数与下层帧的车道数可以并列对比，错误在传参那一行还是除法算式一眼可辨，错误常在上层。看不懂的库函数帧是解释器自己的调用，忽略即可；语法错误在编译期就失败、根本没建帧，堆栈自然为空。

### 5A.7.2 示例代码

**核心要点**：验证堆栈与调用链的一一对应——在 `lane_density` 设断点，堆栈面板自上而下依次是 `lane_density` ← `judge` ← `report` ← `<module>`，点每一帧都能看到该层的局部变量。

In [ ]:
# ---------- 第 05A 节 · 示例 5A-6：调用堆栈，看清"谁调用了谁" ----------
# 对应知识点：5A.6 调用堆栈 Call Stack —— 顺着调用链找到出问题的那一层
# 操作：在 lane_density 的 return 行设断点 → F5 暂停 →
#      「调用堆栈」面板自上而下依次是 lane_density ← judge ← report ← <module>，
#      点任意一帧，编辑器跳到对应代码行，变量面板切换成该层的局部变量。

def lane_density(flow, lanes):
    """单车道车流密度（辆/车道）"""
    return round(flow / lanes)      # ← 断点设在这一行

def judge(flow, lanes):
    """按单车道密度判定服务水平"""
    d = lane_density(flow, lanes)
    if d > 400:
        return "拥堵"
    if d > 250:
        return "缓行"
    return "畅通"

def report(name, flow, lanes):
    print(name, "单车道", round(flow / lanes), "辆 →", judge(flow, lanes))

report("京港澳", 1650, 3)
report("京开", 900, 3)

**操作步骤**：

1. 在 `lane_density` 的 `return` 行设断点，按 `F5`；
2. 看「调用堆栈」：应有 4 帧，点最下面的 `<module>`，光标跳到 `report("京港澳", 1650, 3)`；
3. 点 `report` 帧，看 `name / flow / lanes` 三个形参；
4. 点回 `lane_density` 帧，看这里的 `flow / lanes` 与上一层是否一致；
5. 按 `F5` 到第二次调用（京开），再比对一次。

**运行结果**：

```text
京港澳 单车道 550 辆 → 拥堵
京开 单车道 300 辆 → 缓行
```

### 5A.7.3 练习代码

**任务**：京开断面单车道 300 辆，应判"缓行"，程序却判成"拥堵"。在 `lane_density` 的 `return` 行设断点，按两次 `F5` 停在京开那一次，再到堆栈里比较 `judge` 收到的 `lanes` 与它转交出去的实参。

**操作清单**：

1. 在 `lane_density` 的 `return` 行设断点，按 `F5`（第一次停在京港澳）；
2. 再按 `F5`（第二次停在京开），此时 `flow = 900`；
3. 在堆栈里点 `judge` 帧，记下 `lanes` 的值；
4. 点回 `lane_density` 帧，记下 `lanes` 的值，两者对比；
5. 改 `judge` 里的一处，使京开判为"缓行"、京港澳仍为"拥堵"。

In [ ]:
# ---------- 第 05A 节 · 练习 5A-6：调用堆栈定位"错在哪一层" ----------
# 现象：京开断面单车道 300 辆，应判"缓行"，实际判成"拥堵"
# 任务：在 lane_density 的 return 行设断点 → F5（会停在京港澳那次）→ F5 继续（停在京开那次）
#      → 在「调用堆栈」里点 judge 那一帧，比较 judge 收到的 lanes 与它转交给 lane_density 的实参
# 提示：错误往往不在"算式"，而在"传参"；堆栈面板能同时看到上下两层的变量

def lane_density(flow, lanes):
    return round(flow / lanes)      # ← 断点设在这一行

def judge(flow, lanes):
    d = lane_density(flow, 2)       # 此处传的车道数与形参不一致
    if d > 400:
        return "拥堵"
    if d > 250:
        return "缓行"
    return "畅通"

def report(name, flow, lanes):
    print(name, "→", judge(flow, lanes))

report("京港澳", 1650, 3)
report("京开", 900, 3)

**运行结果**（未修复）：

```text
京港澳 → 拥堵
京开 → 拥堵
```

**自检要点**：

- [ ] 能说出 `judge` 帧里 `lanes = 3`，而 `lane_density` 帧里 `lanes = 2`；
- [ ] 能指出错误在"传参"而不是"除法"；
- [ ] 修复后京港澳仍为"拥堵"、京开变为"缓行"。

**变式代码（迁移练习）**：用 report → judge → density 三层调用把各帧的局部变量直接打印，验证错误在传参而非算式；这是 Notebook 内可运行的对照代码，断点/单步等实际操作仍须在 Trae 中完成。

In [ ]:
# 对照：把「调用堆栈」各帧的局部变量直接打印，看清传参在哪一层被写死
def density(flow, lanes):
    print("  [density 帧] flow =", flow, "lanes =", lanes)
    return round(flow / lanes)
def judge(flow, lanes):
    print(" [judge 帧] flow =", flow, "lanes =", lanes)
    return density(flow, 2)
for name, flow, lanes in [("京港澳", 1650, 3), ("京开", 900, 3)]:
    print("[module 帧]", name, "→ 单车道", judge(flow, lanes))

**预期输出**：

```text
 [judge 帧] flow = 1650 lanes = 3
  [density 帧] flow = 1650 lanes = 2
[module 帧] 京港澳 → 单车道 825
 [judge 帧] flow = 900 lanes = 3
  [density 帧] flow = 900 lanes = 2
[module 帧] 京开 → 单车道 450
```

### 5A.7.4 实操步骤（上机跟随）

1. 运行对照代码，每个断面依次打印 module 帧、judge 帧、density 帧三行，模拟堆栈自上而下的现场。
2. 核对京港澳：judge 帧 lanes = 3，而 density 帧 lanes = 2，两层不一致。
3. 核对京开：同样 judge 帧 3、density 帧 2，单车道结果为 450 而非按 3 车道的 300。
4. 据此判断错误在 judge 里 density(flow, 2) 的传参，而不在除法算式本身。
5. 在 Trae 中于 density 的 return 行设断点，打开调用堆栈点 judge / density 两帧，核对各自 lanes 的值。

### 5A.7.5 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 只看栈顶一帧就下结论 | 错误常在上层函数的传参里 | 逐帧点看，比较上下层同名变量 |
| 堆栈里出现 runpy / ipykernel | 那是解释器或 Notebook 自己的调用帧 | 忽略即可，只看业务函数帧 |
| 语法错误时堆栈是空的 | 编译期就失败、根本没建立栈帧 | 堆栈只在运行暂停或异常时才有内容 |
| 以为堆栈能改历史 | 调用堆栈是只读的现场记录 | 只能查看，不能回改已执行的帧 |
| 点帧后变量面板没变 | 未切换到对应帧 | 确认选中的是目标帧，再看其局部变量 |

## 5A.8 知识点 5A.7　条件断点与日志点

### 5A.8.1 集中讲解

普通断点是"**每次**到这行都停"。循环 12 次就停 12 次，很快就不耐烦——于是有了两种改良：

| 类型 | 怎么设 | 行为 | 适合 |
|---|---|---|---|
| **条件断点** | 右键红点 →「编辑断点」→ 填条件表达式 | **只有条件为真**时才停 | 只关心某一轮 / 某个值满足条件时 |
| **日志点**（Logpoint） | 右键红点 →「编辑断点」→ 选"日志消息" | **不停**，只在调试控制台输出一条消息 | 想知道"每一轮的值"，但不想打断 |

**条件断点的表达式**用**当前作用域**里的变量写，写法与 `if` 后面的条件完全一样：

| 目的 | 条件表达式 |
|---|---|
| 只在第 8 轮停 | `i == 7` |
| 只在流量超阈值时停 | `flows[i] > 1500` |
| 只在遇到异常值时停 | `speeds[i] < 10` |
| 只在特定车辆时停 | `plate == "京A12345"` |

**日志点的消息**可以插值：写 `第 {i+1} 个时段：{flows[i]}`，花括号里放表达式，程序会把结果填进去打印——效果等同于"临时加了一句 `print`"，但**不改源码、调试结束自动失效**，不污染代码。

**顺序很重要**：条件写错时断点"永远不停"，很容易误以为调试器坏了。先把它改回普通断点验证程序确实经过这一行，再改条件——**先证明"能停"，再收窄"何时停"**，这是排查条件断点的标准套路。

**易错点**：

- 条件里用了**还没定义**的变量（如循环里才有的 `f`）→ 断点永不命中。
- 条件写成赋值 `i = 7`（少了等号）→ 语法无效，断点被忽略。
- 日志点把消息写成 `print(...)` → 只会原样打印这几个字符，因为花括号才是插值符。
- 循环里条件断点仍要按 `F5` 才能到下一轮，别以为它"自动跳"。

**原理补充（概念深化）**：条件断点未改 trace 机制，只给断点附加表达式：每次命中该行先用当前帧 locals 求值，为真才挂起、为假静默放行——循环 N 轮仍触发 N 次钩子，只是'停不下来'。这解释了为何条件用了循环才定义的变量、或把 `i == 7` 写成赋值 `i = 7` 会'永远不停'：第 1 轮取不到名字或表达式非法。日志点则不挂起，只把消息里 `{表达式}` 求值插值写进控制台，等价临时 `print` 却不改源码、会话结束即失效。二者前提都是该行被执行到，故须先用普通断点证明能停、再收窄条件。

### 5A.8.2 示例代码

**核心要点**：验证条件断点"只在该停的那一次停"与日志点"不停但留痕"——12 个时段里只有第 8 个超阈值，普通断点要停 12 次，条件断点 `flows[i] > 1500` 只停 1 次，日志点则一次都不停却能把每轮的值打出来。

In [ ]:
# ---------- 第 05A 节 · 示例 5A-7：条件断点与日志点 ----------
# 对应知识点：5A.7 条件断点（只在满足条件时暂停）与日志点（打印但不打断）
# 操作：
#   1) 在 for 行设普通断点 → 会停 12 次，太吵；
#   2) 右键断点 →「编辑断点」→ 条件填 flows[i] > 1500，就只在超阈值那一次停下；
#   3) 把断点改成「日志点」，消息填 第 {i+1} 个时段：{flows[i]}，不暂停也能看到每一轮的值。

flows = [420, 510, 630, 700, 880, 950, 1100, 1650, 1320, 980, 760, 690]
for i in range(len(flows)):
    print("第", i + 1, "个时段：", flows[i], "辆/2h")

print()
for i in range(len(flows)):
    if flows[i] > 1500:
        print("超阈值时段：第", i + 1, "个，", flows[i], "辆/2h")

**操作步骤**：

1. 在 `print("第", i + 1, ...)` 设普通断点，按 `F5`，数一数要按多少次 `F5` 才能跑完（12 次）；
2. 右键该断点 →「编辑断点」→ 条件填 `flows[i] > 1500`，重启调试（`Ctrl+Shift+F5`）；
3. 这次只停 1 次，`i = 7`；
4. 把断点改成日志点，消息填 `第 {i+1} 个时段：{flows[i]}`，重启；
5. 程序不停，但调试控制台逐行打出 12 条记录。

**运行结果**：

```text
第 1 个时段： 420 辆/2h
第 2 个时段： 510 辆/2h
第 3 个时段： 630 辆/2h
第 4 个时段： 700 辆/2h
第 5 个时段： 880 辆/2h
第 6 个时段： 950 辆/2h
第 7 个时段： 1100 辆/2h
第 8 个时段： 1650 辆/2h
第 9 个时段： 1320 辆/2h
第 10 个时段： 980 辆/2h
第 11 个时段： 760 辆/2h
第 12 个时段： 690 辆/2h

超阈值时段：第 8 个， 1650 辆/2h
```

### 5A.8.3 练习代码

**任务**：8 个样本的车速肉眼看去都在 60—82 之间，平均车速却只有 62.1 km/h。用**条件断点**直接停在可疑值上，用**日志点**给每一轮留痕，找出是哪一条数据有问题，并说清它该怎么处理。

**操作清单**：

1. 在 `total = total + speeds[i]` 设普通断点，按 `F5`，数命中断点的次数；
2. 改成条件断点 `speeds[i] < 10`，重启调试，只停 1 次，记下 `i` 与 `speeds[i]`；
3. 改成日志点（消息 `i={i} speed={speeds[i]}`），重启，从调试控制台读出全部 8 条；
4. 回答：平均值偏低的直接原因是什么？这条 `0` 从业务上该怎么看待（提示：传感器丢帧）；
5. 用第 5 章的推导式写一句过滤，把 `0` 值排除后再算平均，结果应为 71.0。

In [ ]:
# ---------- 第 05A 节 · 练习 5A-7：用条件断点揪出异常值 ----------
# 现象：8 个样本的平均车速只有 62.1 km/h，与肉眼看到的 60~82 明显不符
# 任务：
#   1) 在 total = total + speeds[i] 设普通断点，看要停几次；
#   2) 改成条件断点 speeds[i] < 10，只停那一次，看 i 与 speeds[i]；
#   3) 用日志点输出每一轮的 i 与 speeds[i]，确认是哪一条数据有问题；
#   4) 说清这条数据该怎么处理（提示：第 4 章清洗、第 5 章筛选取值）
# 提示：条件断点的表达式用当前作用域里的变量写，不要把 print 写进去

speeds = [78, 65, 71, 0, 82, 69, 74, 58]
total = 0
for i in range(len(speeds)):
    total = total + speeds[i]        # ← 先设普通断点，再改条件断点 speeds[i] < 10

avg = total / len(speeds)
print("样本数：", len(speeds))
print("平均车速：", round(avg, 1), "km/h")

**运行结果**（未修复）：

```text
样本数： 8
平均车速： 62.1 km/h
```

**自检要点**：

- [ ] 条件断点只在 `i = 3` 时停下，`speeds[3]` 为 `0`；
- [ ] 能说出"总和 497 被 8 除"与"被 7 除"的差别（62.1 与 71.0）；
- [ ] 能用 `[s for s in speeds if s > 0]` 这类写法排除异常值。

**变式代码（迁移练习）**：用日志点逐轮打印 12 个时段、用条件表达式筛出唯一超阈值的一轮，验证条件断点只停一次；这是 Notebook 内可运行的对照代码，断点/单步等实际操作仍须在 Trae 中完成。

In [ ]:
# 对照：日志点打印全部轮次，条件断点只在满足 flows[i] > 1500 的那轮"停"
flows = [420, 510, 630, 700, 880, 950, 1100, 1650, 1320, 980, 760, 690]
for i in range(len(flows)):
    print("[日志点] 第", i + 1, "时段：", flows[i])
hit = [i + 1 for i in range(len(flows)) if flows[i] > 1500]
print("[条件断点] 命中的时段编号：", hit, "（仅 1 次）")
print("该时段流量：", flows[hit[0] - 1], "辆/2h | 共扫描", len(flows), "轮")

**预期输出**：

```text
[日志点] 第 1 时段： 420
[日志点] 第 2 时段： 510
[日志点] 第 3 时段： 630
[日志点] 第 4 时段： 700
[日志点] 第 5 时段： 880
[日志点] 第 6 时段： 950
[日志点] 第 7 时段： 1100
[日志点] 第 8 时段： 1650
[日志点] 第 9 时段： 1320
[日志点] 第 10 时段： 980
[日志点] 第 11 时段： 760
[日志点] 第 12 时段： 690
[条件断点] 命中的时段编号： [8] （仅 1 次）
该时段流量： 1650 辆/2h | 共扫描 12 轮
```

### 5A.8.4 实操步骤（上机跟随）

1. 运行对照代码，先数出 12 行『[日志点] 第 n 时段』，确认日志点不打断、逐轮留痕。
2. 看『[条件断点] 命中的时段编号：[8]（仅 1 次）』，确认 flows[i] > 1500 只在第 8 轮成立。
3. 看『该时段流量：1650 辆/2h | 共扫描 12 轮』，确认命中值就是唯一的超阈值样本。
4. 把条件改成 flows[i] > 2000 再跑，命中列表应为空，验证条件写错会『永远不停』。
5. 在 Trae 中先把断点设普通、证明能停，再改条件 flows[i] > 1500，最后改成日志点消息 第 {i+1} 时段：{flows[i]}。

### 5A.8.5 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 条件断点一次都不停 | 条件用了循环里才定义的变量、或永为假 | 先改回普通断点证明能停，再收窄条件 |
| 把 i == 7 写成 i = 7 | 赋值不是合法条件，断点被忽略 | 用 == 比较，别写单个等号 |
| 日志点把 print(...) 原样打出来 | 日志消息用 {表达式} 插值，不写 print | 消息写成 第 {i+1} 时段：{flows[i]} |
| 以为条件断点会自动跳到下一轮 | 条件为真才挂起，仍要按 F5 继续 | 命中后按 F5 前进 |
| 循环 12 轮普通断点停了 12 次太吵 | 普通断点每行都停 | 改用条件断点或日志点 |

## 5A.9 七个技能速查表

| 想做什么 | 用什么 | 快捷键 / 位置 | 一句话记忆 |
|---|---|---|---|
| 让程序停在某行 | 行断点 | `F9` / 点行号左侧 | 红点即"到此一停" |
| 开始调试 | 启动调试 | `F5` | 不是 `Ctrl+F5` |
| 一次走一行，不进函数 | 逐过程 | `F10` | 只看结果，不看过程 |
| 钻进函数 | 步入 | `F11` | 怀疑函数内部 |
| 从函数出来 | 步出 | `Shift+F11` | 看返回值是否对得上 |
| 看变量值 | 变量面板 / 悬停 | 左侧 / 鼠标停留 | 先展开，别只看外层 |
| 看表达式结果 | 监视 | 「监视」区 `+` | 能写调用，不只是名字 |
| 看谁调用了谁 | 调用堆栈 | 「调用堆栈」区 | 错误常在上一层 |
| 只在某次停 | 条件断点 | 右键断点 → 编辑 | 先证明能停，再收窄条件 |
| 不停但留痕 | 日志点 | 右键断点 → 日志消息 | `{表达式}` 是插值 |

# 第二部分　代码示范

## 案例 5A-1　信号配时计算器

- **覆盖知识点**：5A.1（断点与启动）、5A.2（逐过程）、5A.3（步入）、5A.5（变量与监视）
- **源文件**：`case1_信号配时计算器.py`
- **预设 bug**：3 个（全部为"不报错、结果错"的逻辑错误）
- **难度**：基础　**建议用时**：35 分钟（教师演示 20 min + 学生复现 15 min）

### 任务情境

某十字路口设两个相位，周期 90 秒，每相位损失时间 3 秒。控制机要按两相位车流量比分配绿灯并输出绿信比。程序由上一届学生编写，**能跑通但配时明显不合理**：流量大的南北直行只分到 56 秒，流量小的东西直行反而拿到 84 秒。

### 任务要求

1. 先**直接运行**，记录错误输出，写下三条"明显不合理"之处；
2. 用断点 + `F10` 定位"总流量比 Y 偏小"的原因；
3. 用断点 + `F11` + 变量面板定位"绿灯分配反了"的原因；
4. 用监视表达式定位"绿信比之和 ≠ 1"的原因；
5. 修复后运行，校验：绿灯合计 = 84 秒、两个绿信比之和 ≈ 1。

### 输入规格

| 项 | 要求 |
|---|---|
| 数据来源 | 被调试脚本 `case1_信号配时计算器.py`；流量（南北直行 720、东西直行 480）、周期 90、每相位损失 3、饱和流量 1650 均写在源码字面量中 |
| 类型 | 调试输入是操作序列：断点行号加按键；流量与周期为 `int`，流量比与绿信比为浮点 |
| 取值范围 | 断点 4 处（第 24 行累加、第 37 行调用、第 45 行校核、第 50 行绿信比）；按键序列含 `F5`、`F10`、`F11`、`Shift+F11` 共 4 个键 |
| 非法值示例 | 按 `Ctrl+F5` 直接运行导致断点不命中、断点呈灰色空心圆、在循环外设断点却期待看到每轮 `y_total`、把 `F11` 与 `F10` 对调 |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 直接运行输出 9 行，第 3 行与第 7 行为空行；调试面板内容不计入行数 |
| 逐行格式 | 第 1 行为 `总流量比 Y = 0.291`；第 4、5 行以 3 个空格开头；第 8、9 行同样以 3 个空格开头 |
| 精度 / 单位 | 流量比与绿信比保留 3 位小数；绿灯为整数，单位秒 |
| 调试面板 | 在第 24 行第 2 轮暂停时，变量面板应显示 `y_total = 0`；按 `F11` 进入第 15—17 行的 `phase_green` 后，应显示 `y_phase = 84`、`y_total = 0.436`、`cycle_eff = 0.291` |
| 判定方式 | 先与「运行结果（未修复）」逐字比对 9 行；修复后再与「运行结果（修复后）」比对第 6 行的 84 秒与第 8、9 行的绿信比 |

### 参考代码（含 3 个预设 bug）

In [ ]:
# ---------- 第 05A 节 · 案例 5A-1：信号配时计算器（含 3 个预设 bug）----------
# 对应知识点：5A.2 逐过程、5A.3 步入、5A.5 变量与监视
# 功能说明：按两个相位的流量比，把 90 秒周期内的有效绿灯时间分配给各相位，
#           并输出绿信比，用于核对分配是否合理。
# 使用方式：先 Ctrl+F5 直接跑一遍，记下"错的输出"；再按提示设断点逐行定位。
# 判定标准：绿灯合计 = 有效绿灯时间；两个相位绿信比之和 ≈ 1。

CYCLE = 90                 # 信号周期（秒）
LOST_PER_PHASE = 3         # 每相位损失时间（秒）
SATURATION = 1650          # 单车道饱和流量（pcu/h）

flows = {"南北直行": 720, "东西直行": 480}      # 各相位车流量（辆/小时）


def phase_green(y_phase, y_total, cycle_eff):
    """按流量比把有效绿灯时间分配给一个相位"""
    return round(cycle_eff * y_phase / y_total)


# 1) 总流量比 Y = Σ(q / S)
y_total = 0
for name in flows:
    y_total = 0
    y_total = y_total + flows[name] / SATURATION
print("总流量比 Y =", round(y_total, 3))

# 2) 有效绿灯时间 = 周期 − 总损失时间
lost = LOST_PER_PHASE * len(flows)
cycle_eff = CYCLE - lost
print("总损失", lost, "秒，有效绿灯", cycle_eff, "秒")
print()

# 3) 分配各相位绿灯时间
greens = {}
for name in flows:
    y = flows[name] / SATURATION
    g = phase_green(cycle_eff, y, y_total)
    greens[name] = g
    print("  ", name, "流量比", round(y, 3), "→ 绿灯", g, "秒")

# 4) 校核：绿灯合计应等于有效绿灯
total_green = 0
for name in greens:
    total_green = total_green + greens[name]
print("绿灯合计", total_green, "秒（应等于", cycle_eff, "秒）")
print()

# 5) 绿信比 λ = g / C
for name in greens:
    lam = greens[name] / cycle_eff
    print("  ", name, "绿信比 λ =", round(lam, 3))

### 运行结果（未修复）

```text
总流量比 Y = 0.291
总损失 6 秒，有效绿灯 84 秒

   南北直行 流量比 0.436 → 绿灯 56 秒
   东西直行 流量比 0.291 → 绿灯 84 秒
绿灯合计 140 秒（应等于 84 秒）

   南北直行 绿信比 λ = 0.667
   东西直行 绿信比 λ = 1.0
```

### 定位路径（课堂演示脚本，教师照着走）

| 步骤 | 操作 | 观察 | 结论 |
|---|---|---|---|
| 1 | 直接运行，读输出 | Y = 0.291，而两个流量比分别为 0.436 与 0.291 | Y 应**大于**每个分量（它是总和），现在却等于其中一个 |
| 2 | 在 `y_total = y_total + flows[name] / SATURATION` 设断点，`F5` | `y_total = 0`、`flows[name]/SATURATION ≈ 0.436` | 第一轮正常 |
| 3 | 按 `F5` 进第二轮，看**上一行** `y_total = 0` | 上一轮算出的 0.436 消失了，`y_total` 又变 0 | **bug 1**：初始化写在了循环里，每轮清零 |
| 4 | 把该行移出循环，重跑 | Y = 0.727，但绿灯仍是 56 / 84，合计 140 | 修好一个，问题还在——继续 |
| 5 | 在 `g = phase_green(cycle_eff, y, y_total)` 设断点，按 `F11` | 函数里 `y_phase = 84`、`y_total = 0.436`、`cycle_eff = 0.291` | 形参与实参**完全错位**：周期进了"流量比"，流量比进了"周期" |
| 6 | 按实参顺序修正调用（`y, y_total, cycle_eff`） | 绿灯 50 / 34，合计 84 | **bug 2** 修复 |
| 7 | 在 `lam = greens[name] / cycle_eff` 加监视 `greens[name] / CYCLE` | 0.667 与 0.556 相差明显 | **bug 3**：绿信比的分母应是周期 90，不是有效绿 84 |
| 8 | 改为 `greens[name] / CYCLE`，重跑 | λ = 0.556、0.378，合计 0.934 | 全部通过（余下 0.066 为损失时间占比） |

### 修复后的关键改动

```python
# bug 1：初始化移到循环外
y_total = 0
for name in flows:
    y_total = y_total + flows[name] / SATURATION

# bug 2：实参顺序与形参 (y_phase, y_total, cycle_eff) 对齐
g = phase_green(y, y_total, cycle_eff)

# bug 3：绿信比的分母是周期 C，不是有效绿灯
lam = greens[name] / CYCLE
```

### 运行结果（修复后）

```text
总流量比 Y = 0.727
总损失 6 秒，有效绿灯 84 秒

   南北直行 流量比 0.436 → 绿灯 50 秒
   东西直行 流量比 0.291 → 绿灯 34 秒
绿灯合计 84 秒（应等于 84 秒）

   南北直行 绿信比 λ = 0.556
   东西直行 绿信比 λ = 0.378
```

### 代码解读

| 代码段 | 对应技能 | 在本案例中的作用 |
|---|---|---|
| `y_total = 0` 的位置 | 5A.2 `F10` | 位置错了就"每轮归零"，是累加类 bug 的头号来源 |
| `phase_green(y_phase, y_total, cycle_eff)` | 5A.3 `F11` | 三个形参都是数值，传错顺序**不会报错**，只能靠进入函数看形参 |
| `greens[name] / cycle_eff` | 5A.5 监视 | 分子分母都是"秒"，量纲合法，只能靠"λ 之和应为 1"这个业务约束发现 |
| `round(cycle_eff * y_phase / y_total)` | 5A.5 变量 | 单看这一行没错，错在"喂给它的三个数" |

**改一改**：把 `round()` 去掉，看绿灯合计是否还等于 84——这会引出"四舍五入后的余数怎么处理"（配时工程里的"绿时补偿"），可留作拓展讨论。

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| 按 `F5` 后程序直接跑完、未停在断点行 | 失败：断点未命中，多因按了 `Ctrl+F5` 或断点呈灰色空心圆 | 看行号左侧是否为红色实心圆，并确认状态栏已选解释器 |
| 第 2 轮暂停时变量面板显示 `y_total = 0.436` | 失败：变量值与"每轮清零"的现象不符，说明 bug 1 已被提前修掉 | 展开变量面板看 `y_total`，与定位路径第 3 步的 0 对照 |
| 按 `F11` 后 `phase_green` 内显示 `y_phase = 84`、`y_total = 0.436` | 不是失败，而是命中 bug 2 的错位特征；此时若判为正常则失败 | 对照形参表逐个核对三个数值的来源 |
| 修复后第 6 行仍为 `绿灯合计 140 秒` | 失败：只改了一处，实参顺序未调整 | 数第 6 行的合计秒数 |
| 修复后第 8、9 行绿信比之和为 1.667 | 失败：分母仍在用有效绿灯 84 而非周期 90 | 把两行小数相加，看是否在 1 附近 |

## 案例 5A-2　早高峰断面统计与告警

- **覆盖知识点**：5A.2（逐过程）、5A.3（步入）、5A.4（步出）、5A.5（变量与监视）、5A.7（条件断点）
- **源文件**：`case2_早高峰统计.py`
- **预设 bug**：4 个
- **难度**：综合　**建议用时**：45 分钟

### 任务情境

早高峰结束后，检测中心导出 9 条过车记录（元组：车牌、断面、平均行程车速），需要按断面汇总平均车速、给出拥堵判定，并统计涉及车辆数与高速车辆清单。程序不报错，报表却不对：断面平均车速只有 3.7—6.4 km/h，三个断面全判「缓行」，涉及车辆数写成 9 辆（实际只有 7 辆不同车牌）。

### 任务要求

1. 先运行，把"不合理"的地方逐条写下来（至少 4 条）；
2. 定位"平均车速只有几 km/h"的原因，并说清它由**两个**独立错误叠加而成；
3. 定位"机场断面明明 35.3 km/h 却判缓行"的原因；
4. 定位"涉及车辆数偏大"的原因，并说明要用第 5 章的哪个知识点修正；
5. 修复后校验：三断面平均车速为 45.0 / 60.0 / 35.3，判定为 缓行 / 畅通 / 拥堵，涉及车辆 7 辆。

### 输入规格

| 项 | 要求 |
|---|---|
| 数据来源 | 被调试脚本 `case2_早高峰统计.py`；9 条过车记录（车牌、断面、平均行程车速）写在源码字面量中 |
| 类型 | 每条记录为 3 元素元组：车牌 `str`、断面 `str`、车速 `int`；调试输入为断点行号加按键，条件断点为布尔表达式 |
| 取值范围 | 记录 9 条、断面 3 个、车速 33—62；断点 4 处（第 33 行收集、第 24 行返回、第 40 行判定、第 52 行计数）；按键序列含 `F5`、`F10`、`F11`、`Shift+F11` |
| 非法值示例 | 用 `Ctrl+F5` 直接运行、断点设在空行、条件断点里写 `print(avg)`、把 `F10` 与 `F11` 对调 |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 未修复时输出 8 行；第 2 行与第 6 行为 40 个 `-` |
| 逐行格式 | 第 1 行为「断面数： 3」与「采样记录： 9 条」两部分；第 3—5 行依次为京港澳、京开、机场三条断面行 |
| 精度 / 单位 | 平均车速保留 1 位小数，单位 km/h；记录条数为整数；等级为 缓行、畅通、拥堵 三个词之一 |
| 调试面板 | 在第 33 行停第 3 次时 `speeds["京港澳"]` 应显示 `[39]`；按 `F11` 进入第 22—24 行的 `avg_speed` 后，`len(records)` 应为 9、`sum(speeds)` 应为 273 |
| 判定方式 | 先与「运行结果（未修复）」逐字比对 8 行；修复后再与「运行结果（修复后）」比对第 3—5 行的 45.0、60.0、35.3 |

### 参考代码（含 4 个预设 bug）

In [ ]:
# ---------- 第 05A 节 · 案例 5A-2：早高峰断面统计与告警（含 4 个预设 bug）----------
# 对应知识点：5A.2 逐过程、5A.3 步入、5A.5 变量与监视、5A.7 条件断点
# 数据结构：records 每条为元组 (车牌, 断面, 平均行程车速 km/h)，同一车牌可能被多次检测
# 判定标准：
#   1) 断面平均车速应在 30~60 km/h 之间，与记录里的单车车速同量级；
#   2) 平均车速 < 40 判"拥堵"，40~59 判"缓行"，≥ 60 判"畅通"；
#   3) 涉及车辆数按车牌去重后统计。

records = [
    ("京A12345", "京港澳", 48),
    ("京B67890", "京港澳", 42),
    ("京A12345", "京港澳", 51),
    ("津C24680", "京港澳", 39),
    ("京E11223", "京开", 62),
    ("京F33445", "京开", 58),
    ("京G55667", "机场", 35),
    ("京G55667", "机场", 38),
    ("京H77889", "机场", 33),
]


def avg_speed(speeds):
    """断面平均行程车速（km/h）"""
    return round(sum(speeds) / len(records), 1)


count = {}
speeds = {}
for i in range(len(records)):
    plate = records[i][0]
    road = records[i][1]
    count[road] = count.get(road, 0) + 1
    speeds[road] = [records[i][2]]

print("断面数：", len(count), "| 采样记录：", len(records), "条")
print("-" * 40)

for road in count:
    avg = avg_speed(speeds[road])
    if avg < 60:
        level = "缓行"
    elif avg < 40:
        level = "拥堵"
    else:
        level = "畅通"
    print(road, "→ 平均", avg, "km/h |", count[road], "条记录 |", level)

print("-" * 40)
plates = []
for i in range(len(records)):
    plates.append(records[i][0])
print("涉及车辆数：", len(records), "辆（含重复检测）")

fast = [records[i][0] for i in range(len(records)) if records[i][2] > 60]
print("车速高于 60 km/h 的车辆：", fast, "共", len(fast), "辆")

### 运行结果（未修复）

```text
断面数： 3 | 采样记录： 9 条
----------------------------------------
京港澳 → 平均 4.3 km/h | 4 条记录 | 缓行
京开 → 平均 6.4 km/h | 2 条记录 | 缓行
机场 → 平均 3.7 km/h | 3 条记录 | 缓行
----------------------------------------
涉及车辆数： 9 辆（含重复检测）
车速高于 60 km/h 的车辆： ['京E11223'] 共 1 辆
```

### 定位路径

| 步骤 | 操作 | 观察 | 结论 |
|---|---|---|---|
| 1 | 在 `avg = avg_speed(speeds[road])` 设断点，`F5` | 展开 `speeds`，`speeds["京港澳"]` 只有 `[39]` 一个元素 | 收集环节就错了，先别急着看平均 |
| 2 | 回到收集循环，在 `speeds[road] = [records[i][2]]` 设断点连按 `F10` | 每轮都把整个列表**覆盖**成单元素列表 | **bug 1**：应是"追加"，不是"重新赋值" |
| 3 | 改为先建空列表再 `append`，重跑 | 平均变成 30.3 / 13.3 / 11.8，仍不合理 | 第二个错误浮出来 |
| 4 | 在 `avg_speed` 的 `return` 设断点，`F11` 进入 | 函数里 `sum(speeds) = 273`、`len(records) = 9` | **bug 2**：分母用了全局记录总数，不是本断面条数（也可用 `Shift+F11` 跳出后看返回值） |
| 5 | 改为 `len(speeds)`，重跑 | 45.0 / 60.0 / 35.3，机场仍判"缓行" | 数值对了，判定还错 |
| 6 | 在 `if avg < 60:` 设**条件断点** `avg < 40` | 断点**不命中**，说明 `avg < 40` 的分支永远走不到 | **bug 3**：分支顺序反了，宽条件在前把窄条件吃掉 |
| 7 | 改为先判 `avg < 40` 再判 `avg < 60` | 机场判"拥堵" | 判定正确 |
| 8 | 看 `plates` 列表与 `len(records)` | `plates` 有 9 项，但不同车牌只有 7 个 | **bug 4**：应 `len(set(plates))` 去重（知识点 5.4） |

> **教学要点**：bug 1 与 bug 2 会**互相掩盖**。修好 bug 1 后平均车速变成 30.3，看着"像那么回事"，很容易以为改完了；只有拿"断面平均车速应落在 30—60"这条业务常识去卡，才会发现还有一层。这正是"一次只改一处、改完必须重跑"的意义。

### 修复后的关键改动

```python
# bug 1：先建空列表再追加，不要每轮重新赋值
if road not in speeds:
    speeds[road] = []
speeds[road].append(records[i][2])

# bug 2：分母是本断面的条数
return round(sum(speeds) / len(speeds), 1)

# bug 3：窄条件在前，宽条件在后
if avg < 40:
    level = "拥堵"
elif avg < 60:
    level = "缓行"
else:
    level = "畅通"

# bug 4：按车牌去重后再计数
print("涉及车辆数：", len(set(plates)), "辆（已按车牌去重）")
```

### 运行结果（修复后）

```text
断面数： 3 | 采样记录： 9 条
----------------------------------------
京港澳 → 平均 45.0 km/h | 4 条记录 | 缓行
京开 → 平均 60.0 km/h | 2 条记录 | 畅通
机场 → 平均 35.3 km/h | 3 条记录 | 拥堵
----------------------------------------
涉及车辆数： 7 辆（已按车牌去重）
车速高于 60 km/h 的车辆： ['京E11223'] 共 1 辆
```

### 代码解读

| 代码段 | 对应技能 | 在本案例中的作用 |
|---|---|---|
| `speeds[road] = [records[i][2]]` | 5A.2 `F10` | 赋值与追加只差一个词，结果差一个量级 |
| `len(records)` 出现在函数里 | 5A.4 步出 / 5A.5 监视 | 函数"偷用"了全局变量，看形参发现不了，看函数体才行 |
| `if avg < 60: ... elif avg < 40:` | 5A.7 条件断点 | 用条件断点验证"这个分支走不到"，比读代码快 |
| `plates` 建了却没用 | 5A.5 变量 | **未被使用的变量**本身就是 bug 的线索 |
| `round(sum(speeds) / len(speeds), 1)` | 第 5 章统计 | 平均值 = 总和 ÷ 个数，"个数"指本组个数 |

**改一改**：把判定阈值改成"拥堵 < 30、缓行 < 50"，再跑一遍——机场 35.3 从"拥堵"变"缓行"。借此讨论"阈值是业务参数，不该散落在代码里"（第 10 章会讲配置与模块）。

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| 按 `F5` 后未停在 `avg = avg_speed(speeds[road])` | 失败：断点未命中 | 查断点颜色与调试启动方式 |
| 变量面板显示 `speeds["京港澳"]` 有 4 个元素 | 失败：与 `[39]` 单元素的现象不符，说明 bug 1 已被修掉 | 展开 `speeds` 后数该键的元素个数 |
| 条件断点 `avg < 40` 在机场断面命中 | 失败：未修复时该分支不应走到，说明分支顺序已被改动 | 看条件断点是否命中，并比对第 3—5 行的等级 |
| 修复后第 5 行仍为 `机场 → 平均 11.8 km/h` | 失败：只改了收集，未改分母 | 读第 5 行平均车速的数值 |
| 修复后第 7 行仍为 9 辆 | 失败：未按车牌去重 | 看第 7 行数字与 `len(set(plates))` 的差 |

---

# 第三部分　布置练习

## 练习 5A-1　ETC 计费核对

- **对应知识点**：5A.1（断点）、5A.2（逐过程）、5A.5（变量与监视）、5A.7（条件断点）
- **提交文件**：`practice1_ETC计费核对.py`（在原文件上修改，不要另存）
- **预设 bug**：3 个
- **难度**：基础　**建议用时**：30 分钟

### 任务情境

你在收费站稽核室核对昨日 ETC 流水：4 笔记录按「费率 × 里程」计费，超出 100 km 打 8 折，再乘会员折扣。程序不报错，普通卡两笔却全算成 0.00 元、合计也是 0.00 元，大型车 140 km 那笔还偏低。你要用断点与监视表达式分清症状与病根，把合计改到 322.21 元。

### 题目

收费站按"费率 × 里程"计费，超过 100 km 的部分打 8 折，再乘会员折扣（金卡 85 折、银卡 95 折、普通卡不打折）。程序对 4 笔通行记录算账，**普通卡车辆全部算成 0.00 元，合计也是 0.00 元**；大型车 140 km 的金额也偏低。

### 要求

1. 先运行，写下"应收 0.00 元"与"合计 0.00 元"这两处异常，判断哪个是**症状**、哪个是**病根**；
2. 用 `F10` 逐行跟住 `total`，定位合计为 0 的原因；
3. 用监视表达式观察 `DISCOUNT.get(level, 0)` 的返回值，定位普通卡为 0 元的原因；
4. 用断点 + 变量面板核对 `fee` 的三段计算，定位"超过 100 km 部分"算错的原因；
5. 修复后校验：4 笔金额为 32.90 / 158.40 / 46.31 / 84.60，合计 **322.21 元**。

### 提示

- 「症状」是能直接看到的现象，「病根」是造成现象的代码。合计为 0 很可能是"每笔都算成了 0"或"合计只保留了最后一笔"，两种病因的修法完全不同——先分清。
- `dict.get(键, 默认值)` 的默认值只在**键不存在**时才生效。想一想："普通"这个键在 `DISCOUNT` 里存在吗？
- 分段计费的算式是"前 100 km 原价 + 超出部分打折"，超出部分是 `km - 100`，不是 `km`。
- 建议先修"合计"，再修"折扣"，最后修"分段"——**一次只改一处，改完必须重跑**。

### 自检要点（不含答案）

- [ ] 合计为 322.21 元，且 4 笔金额之和与之相等；
- [ ] 普通卡两笔分别为 158.40 与 84.60，不再是 0.00；
- [ ] 金卡 86 km 仍为 32.90（不足 100 km，不受分段影响）；
- [ ] 能说出 3 个 bug 分别用哪个技能定位的。

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 在原文件 `practice1_ETC计费核对.py` 上修完 3 处后运行 | 8 行输出，4 笔金额 32.90 / 158.40 / 46.31 / 84.60，合计 322.21 元 | 与答案 5A-1「运行结果（修复后）」逐字比对 8 行 |
| T2 | 边界 | 把大型车里程由 140 改为恰好 100 | 第 4 行金额变为 120.00，合计 283.81 元 | 核对第 4 行金额与末行合计 |
| T3 | 异常 | 把第 1 条记录的车型改为字典里没有的 `特大型车` | 抛 `KeyError: '特大型车'`，输出止于分隔线 | 看报错信息含 `KeyError` 与车型名 |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| bug 1 定位与修复 | 运行并读末行 | 合计 322.21 元，`total = 0` 位于循环外 | 3 |
| bug 2 定位与修复 | 读第 4、6 行 | 普通卡两笔为 158.40 与 84.60 | 3 |
| bug 3 定位与修复 | 读第 4 行 | 140 km 金额为 158.40，基数为 `km - 100` | 2 |
| 调试过程记录 | 查提交的笔记 | 写出每个 bug 用的技能与断点行 | 2 |
| 合计 | — | — | 10 |

## 练习 5A-2　分断面告警报表

- **对应知识点**：5A.5（变量与监视）、5A.6（调用堆栈）、5A.7（条件断点）、第 5 章字典与去重
- **提交文件**：`practice2_拥堵告警.py`（在原文件上修改）
- **预设 bug**：4 个
- **难度**：综合　**建议用时**：40 分钟

### 任务情境

你在路网监测中心复核早高峰报表：9 条过车记录要按断面汇总平均车速、条数与拥堵判定，并列出低速车辆清单。程序不报错，报表却不对：机场北与机场南被并成「机场」，京开平均车速写成 40.0，两个机场断面都判缓行，35 km/h 的车也被收进清单。你得用监视表达式、条件断点与调用堆栈找出这 4 处。

### 题目

9 条过车记录需按断面汇总，输出平均车速、条数与拥堵判定，并列出低速车辆清单。程序输出的报表有四处不对：机场北与机场南被合成一条"京港"以外的"机场"；京开平均车速 40.0；机场两个断面都判"缓行"（实际应"拥堵"）；低速清单里多了一辆恰好 35 km/h 的车。

### 要求

1. 运行并对照"判定标准"，逐条写出 4 处异常；
2. 定位"机场北 / 机场南被合并"的原因（提示：看分组键是怎么来的）；
3. 定位"京开平均车速 40.0"的原因（提示：分母是固定的还是算出来的）；
4. 定位"机场两个断面都判缓行"的原因（提示：分支顺序）；
5. 定位"35 km/h 的车被误收进低速清单"的原因（提示：边界该不该取等号）；
6. 修复后校验：4 个断面为 京港澳 47.0 缓行 / 京开 60.0 畅通 / 机场北 35.5 拥堵 / 机场南 30.0 拥堵，低速车辆 2 辆。

### 提示

- 分组键用切片 `records[i][1][:2]` 会**截断**断面名，两个不同断面可能落到同一个键上——用监视表达式 `records[i][1]` 与 `road` 对比即可发现。
- 平均值的分母写成常数（如 `3`）会"碰巧"对一两个断面，这是最阴的 bug：**看着对，其实是巧合**。
- 判断分支要从"最严格"往"最宽松"写；先判宽条件，窄条件永远走不到。
- "低于 35"与"不超过 35"是两回事，边界值 35 该不该收，取决于业务口径，**代码必须与口径一致**。
- 若某处一时看不出，用**条件断点**把 `road` 或 `avg` 的值卡在某一个断面上再停。

### 自检要点（不含答案）

- [ ] 报表出现 4 行，机场北与机场南各占一行；
- [ ] 京开平均车速 60.0，判定"畅通"；
- [ ] 机场北 35.5、机场南 30.0，均判"拥堵"；
- [ ] 低速清单为 2 辆，不含 35 km/h 的那辆；
- [ ] 能说出 4 个 bug 中哪个是"边界"问题、哪个是"顺序"问题。

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 在原文件 `practice2_拥堵告警.py` 上修完 4 处后运行 | 8 行输出，4 个断面平均 47.0 / 60.0 / 35.5 / 30.0，低速车辆 2 辆 | 与答案 5A-2「运行结果（修复后）」逐字比对 8 行 |
| T2 | 边界 | 把机场南 29 改为 35 | 第 5 行平均 33.0 仍判拥堵，第 7 行低速车辆降为 1 辆 | 核对第 5 行平均车速与第 7 行辆数 |
| T3 | 异常 | 删掉 `if road not in roads:` 那段初始化 | 抛 `KeyError: '京港澳'`，无报表输出 | 看报错信息含 `KeyError` 与首个断面名 |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| 分组键修复 | 数报表行数 | 4 个断面各占一行，机场北与机场南分开 | 3 |
| 平均分母修复 | 读 4 个平均值 | 47.0、60.0、35.5、30.0 逐个相符 | 3 |
| 分支顺序修复 | 读机场两行等级 | 机场北与机场南都判拥堵 | 2 |
| 边界修复 | 读第 7 行辆数 | 低速车辆 2 辆，35 km/h 不在清单内 | 2 |
| 合计 | — | — | 10 |

---

# 第四部分　习题答案

## 答案 5A-1　ETC 计费核对

> 对应 **练习 5A-1**（第三部分）

### bug 清单

| 编号 | 位置 | 现象 | 根因 | 定位技能 |
|---|---|---|---|---|
| 1 | 循环内 `total = 0` | 合计 0.00 | 累加器在循环内被重置，每轮从 0 开始 | 5A.2 `F10` |
| 2 | `DISCOUNT.get(level, 0)` | 普通卡应收 0.00 | `DISCOUNT` 里没有"普通"键，默认值给成 `0` 等于打 0 折 | 5A.5 监视表达式 |
| 3 | `rate * 0.8 * km` | 大型车 140 km 金额偏低 | 打折部分应只算超出 100 km 的里程 `km - 100` | 5A.5 变量面板 / 5A.7 条件断点 |

### 修复后的关键改动

```python
# bug 1：累加器初始化移到循环外（循环前 total = 0，循环内只做加法）
for i in range(len(trips)):
    ...
    fee = fee * DISCOUNT.get(level, 1.0)     # bug 2：默认值应为 1.0（不打折）
    total = total + fee

# bug 3：只对超出 100 km 的部分打折
if km > 100:
    fee = rate * 100 + rate * 0.8 * (km - 100)
else:
    fee = rate * km
```

### 参考代码（完整修复版）

In [ ]:
# ---------- 第 05A 节 · 练习 5A-1 参考答案：ETC 计费核对（3 处 bug 已修复）----------
# 对应知识点：5A.2 逐过程（F10）、5A.5 变量与监视、5A.7 条件断点
# 修复点：
#   ① 累加器 total = 0 原本写在循环内，导致合计只剩最后一笔 → 移到循环外；
#   ② 折扣取值 DISCOUNT.get(level, 0) → get(level, 1.0)，普通卡不再打 0 折；
#   ③ 分段计费基数 rate * 0.8 * km → rate * 0.8 * (km - 100)，只对超出部分打折。
# 校验结果：4 笔金额 32.90 / 158.40 / 46.31 / 84.60，合计 322.21 元。

RATES = {"小型车": 0.45, "中型车": 0.75, "大型车": 1.20}      # 元/km
DISCOUNT = {"金卡": 0.85, "银卡": 0.95}                       # 普通卡不打折

trips = [
    ("京A12345", "小型车", 86, "金卡"),
    ("京B67890", "大型车", 140, "普通"),
    ("津C24680", "中型车", 65, "银卡"),
    ("冀D11223", "小型车", 210, "普通"),
]

print("—— 计费明细 ——")
print("-" * 40)

total = 0                                    # 修复 ①：累加器初始化放在循环外
for i in range(len(trips)):
    plate = trips[i][0]
    kind = trips[i][1]
    km = trips[i][2]
    level = trips[i][3]

    rate = RATES[kind]
    if km > 100:
        fee = rate * 100 + rate * 0.8 * (km - 100)  # 修复 ③：只对超出 100 km 的部分打折
    else:
        fee = rate * km

    fee = fee * DISCOUNT.get(level, 1.0)      # 修复 ②：普通卡默认值为 1.0（不打折）
    total = total + fee                       # 修复 ①：累加器已在循环外初始化，这里只做加法

    print(plate, kind, str(km) + " km", "→", "{:.2f} 元".format(fee))

print("-" * 40)
print("合计：{:.2f} 元".format(total))

### 运行结果（修复后）

```text
—— 计费明细 ——
----------------------------------------
京A12345 小型车 86 km → 32.90 元
京B67890 大型车 140 km → 158.40 元
津C24680 中型车 65 km → 46.31 元
冀D11223 小型车 210 km → 84.60 元
----------------------------------------
合计：322.21 元
```

### 解析

1. **先分症状与病根**：合计 0.00 有两个可能病因——"每笔都算成 0"或"合计只留最后一笔"。看明细就能分辨：明细里前两笔有金额（32.90、46.31），说明单笔计算没问题，**合计被重置**才是病根。这个"先看明细再看合计"的顺序，能省掉一半调试时间。
2. **`get()` 的默认值方向**：折扣的"无折扣"是 `1.0`（乘 1 不变），不是 `0`（乘 0 归零）。默认值该给"业务上的中性值"，给错方向会让缺失数据变成灾难——这与第 5 章"`get()` 做缺失保护"是同一件事，只是默认值的取值要靠业务判断。
3. **分段计费的写法**：`rate * 100 + rate * 0.8 * (km - 100)`。用监视表达式把 `km`、`km - 100`、`fee` 并列观察，能立刻看出打折的基数是不是"超出部分"。
4. **验算**：86×0.45×0.85 = 32.895 → 32.90；(100×1.2 + 40×1.2×0.8) = 158.40；65×0.75×0.95 = 46.3125 → 46.31；(100×0.45 + 110×0.36) = 84.60；合计 322.21。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 合计只等于最后一笔 | 把 `total = 0` 留在循环里 | 移到循环外 |
| 普通卡变成免费 | `get(level, 0)` | 默认值改 `1.0` |
| 140 km 算成 158.40 以外的数 | 打折基数写成 `km` | 改 `(km - 100)` |
| 修改后金额对不上 322.21 | 只改了一部分，没重跑 | 一次改一处，改完必重跑 |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| bug 1 定位与修复 | 3 | 能说出"初始化在循环内"，改动位置正确 |
| bug 2 定位与修复 | 3 | 能解释 `get` 默认值的含义，改为 1.0 |
| bug 3 定位与修复 | 2 | 打折基数为 `km - 100` |
| 调试过程记录 | 2 | 能写出用哪个技能发现的哪个 bug |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 8 行输出与期望逐字一致，合计 322.21 元 | 正常路径，直接复现修复版 |
| T2 | 100 km 不触发分段，金额 120.00 | 判断式为 `km > 100`，取严格大于 |
| T3 | 车型键不存在时抛 `KeyError` | `RATES[kind]` 未做缺失保护，报错即预期结论 |

## 答案 5A-2　分断面告警报表

> 对应 **练习 5A-2**（第三部分）

### bug 清单

| 编号 | 位置 | 现象 | 根因 | 定位技能 |
|---|---|---|---|---|
| 1 | `road = records[i][1][:2]` | 机场北/机场南被合并 | 切片截断了断面名，两个断面落到同一个键 | 5A.5 监视 `records[i][1]` 对比 `road` |
| 2 | `sum(speed_list) / 3` | 京开平均 40.0 | 分母写成常数 3，与条数无关 | 5A.5 监视 `len(speed_list)` |
| 3 | `if avg < 60: ... elif avg < 40:` | 机场两断面都判缓行 | 宽条件在前，`avg < 40` 分支永远走不到 | 5A.7 条件断点 `avg < 40` 不命中 |
| 4 | `records[i][2] <= 35` | 35 km/h 的车被收进低速清单 | 边界取等号，与"低于 35"的口径不符 | 5A.1 断点看该条记录的取值 |

### 修复后的关键改动

```python
# bug 1：断面名完整保留，不做切片
road = records[i][1]

# bug 2：分母用该断面的实际条数
avg = round(sum(speed_list) / len(speed_list), 1)

# bug 3：窄条件在前
if avg < 40:
    level = "拥堵"
elif avg < 60:
    level = "缓行"
else:
    level = "畅通"

# bug 4：严格小于，不含 35
if records[i][2] < 35:
    slow.append(records[i][0])
```

### 参考代码（完整修复版）

In [ ]:
# ---------- 第 05A 节 · 练习 5A-2 参考答案：分断面告警报表（4 处 bug 已修复）----------
# 对应知识点：5A.5 变量与监视、5A.6 调用堆栈、5A.7 条件断点、第 5 章字典与集合去重
# 修复点：
#   ① 分组键 records[i][1][:2] 截断了断面名 → 改用完整名称 records[i][1]；
#   ② 平均车速分母写成常数 3 → 改为 len(speed_list)；
#   ③ 分支顺序宽条件在前 → 改为先判 avg < 40 再判 avg < 60；
#   ④ 低速清单边界 <= 35 → 改为 < 35（业务口径为"低于 35"）。
# 校验结果：4 个断面平均车速 47.0 / 60.0 / 35.5 / 30.0，判定 缓行 / 畅通 / 拥堵 / 拥堵。

records = [
    ("京A12345", "京港澳", 48),
    ("京B67890", "京港澳", 41),
    ("津C24680", "京港澳", 52),
    ("京E11223", "京开", 63),
    ("京F33445", "京开", 57),
    ("京G55667", "机场北", 36),
    ("京H77889", "机场北", 35),
    ("京J99001", "机场南", 29),
    ("京K22110", "机场南", 31),
]

roads = {}
for i in range(len(records)):
    road = records[i][1]                       # 修复 ①：断面名完整保留，不做切片
    if road not in roads:
        roads[road] = []
    roads[road].append(records[i][2])

print("-" * 40)

for road in roads:
    speed_list = roads[road]
    avg = round(sum(speed_list) / len(speed_list), 1)   # 修复 ②：分母用本断面条数
    if avg < 40:                             # 修复 ③：窄条件在前
        level = "拥堵"
    elif avg < 60:
        level = "缓行"
    else:
        level = "畅通"
    print(road, "→ 平均", avg, "km/h |", len(speed_list), "条 |", level)

print("-" * 40)
slow = []
for i in range(len(records)):
    if records[i][2] < 35:                     # 修复 ④：严格小于，不含 35
        slow.append(records[i][0])
print("低速车辆（< 35 km/h）：", slow, "共", len(slow), "辆")

print("涉及断面：", list(roads.keys()))

### 运行结果（修复后）

```text
----------------------------------------
京港澳 → 平均 47.0 km/h | 3 条 | 缓行
京开 → 平均 60.0 km/h | 2 条 | 畅通
机场北 → 平均 35.5 km/h | 2 条 | 拥堵
机场南 → 平均 30.0 km/h | 2 条 | 拥堵
----------------------------------------
低速车辆（< 35 km/h）： ['京J99001', '京K22110'] 共 2 辆
涉及断面： ['京港澳', '京开', '机场北', '机场南']
```

### 解析

1. **切片截断是"键"的问题，不是"值"的问题**：`records[i][1][:2]` 对"京港澳"得到"京港"，对"机场北"与"机场南"都得到"机场"——后者把两个断面合并成一条，条数从 2+2 变成 4，平均值自然也被搅在一起。用监视把原值与键值并列，一眼就能看出来。
2. **常数分母的"巧合"**：分母写 3 时，京港澳恰好是 3 条，算出来 47.0 完全正确——这种"部分正确"最迷人，容易让人以为"已经好了"。**判断标准是"每个断面都要对"，不是"至少有一个对"**。
3. **分支顺序**：`if avg < 60` 先成立后，`elif avg < 40` 永远轮不到，所以 30.0 与 35.5 都被判成"缓行"。用条件断点 `avg < 40` 验证"这个分支不会被执行"，比读代码更有说服力。
4. **边界口径**：`< 35` 与 `<= 35` 差一辆车。凡是"以上 / 以下 / 超过 / 低于"的描述，都要落到代码里的 `>` `>=` `<` `<=` 上逐个核对——这是需求评审阶段就该确认的事。
5. **去重与分组同源**：本练习的分组键错、案例 5A-2 的计数错，都属于"用错了容器的能力"（第 5 章：字典按唯一键分组、集合去重）。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 报表仍只有 3 行 | 只改了打印，没改分组键 | 改 `road = records[i][1]` |
| 京港澳对了、京开仍错 | 分母仍写常数 | 用 `len(speed_list)` |
| 机场北判"缓行" | 分支顺序没调 | 窄条件在前 |
| 低速清单 3 辆 | 边界仍是 `<=` | 改 `<` |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 分组键修复 | 3 | 4 个断面各自成行 |
| 平均分母修复 | 3 | 4 个平均值分别为 47.0 / 60.0 / 35.5 / 30.0 |
| 分支顺序修复 | 2 | 机场两断面判"拥堵" |
| 边界修复 | 2 | 低速清单 2 辆 |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 8 行输出与期望一致，低速车辆 2 辆 | 正常路径，直接复现修复版 |
| T2 | 35 km/h 不入低速清单，机场南均值 33.0 | 判断式为 `< 35`，取严格小于 |
| T3 | 缺初始化时抛 `KeyError` | 修复 ① 之前的循环即建空列表，先建后追加 |

## 教师用：知识点练习代码 bug 速查

> 仅供教师备课与批改使用，学生做完再看。

| 练习代码 | 预设 bug | 修复要点 | 期望输出 |
|---|---|---|---|
| `练习5A1_断点与启动调试.py` | `speeds[-1]` 当成最小值 | 改 `speeds[0]` / `speeds[-1]` 对调 | 最低 45、最高 78 |
| `练习5A2_逐过程执行.py` | `total = 0` 写在循环内 | 移到循环外 | 总流量 2680、平均 893.3 |
| `练习5A3_步入函数.py` | 实参顺序 `(minutes, vehicles)` 与形参反 | 改回 `(vehicles, minutes)` | 144 / 120 / 200 |
| `练习5A4_步出函数.py` | 函数缺少 `return total` | 补 `return total` | 总流量 2680 |
| `练习5A5_变量与监视.py` | `len(road_flows)` 用了字典长度 | 改 `len(flows)` | 3 与 2 |
| `练习5A6_调用堆栈.py` | `lane_density(flow, 2)` 车道数写死 | 改传 `lanes` | 拥堵 / 缓行 |
| `练习5A7_条件断点与日志点.py` | `speeds` 含 0 值未清洗 | 过滤 0 值后再求平均 | 平均 71.0 |

---

## 本节小结

- **调试的本质是观察，不是修改**：断点让你停在可疑处，单步让你看清每一步，变量与堆栈让你看清"此刻的全部现场"。
- **四个动作对应四问**：停哪（`F9`）、怎么走（`F10` / `F11` / `Shift+F11`）、看什么（变量 / 监视 / 堆栈）、何时停（条件断点 / 日志点）。
- **`F10` 与 `F11` 的分界**：只关心结果用 `F10`，怀疑函数内部用 `F11`，看够了用 `Shift+F11` 回家。
- **三类最容易被调试器抓住的 bug**：累加器位置错（每轮清零）、传参顺序错（不报错只算错）、分支顺序错（窄条件被宽条件吃掉）。
- **一次只改一处，改完必重跑**：多个 bug 会互相掩盖，改一处跑一次，才能知道"到底是谁的功劳"。
- **先写预期，再看实际**：没有预期结果的调试，只是在"逛代码"。

## 自检清单

- [ ] 能在任意一行设断点，并用 `F5` 启动调试（不是 `Ctrl+F5` 运行）；
- [ ] 能说出 `F10`、`F11`、`Shift+F11` 的区别，并在需要时正确选用；
- [ ] 会展开变量面板里的字典与列表，会用鼠标悬停看单值；
- [ ] 会添加监视表达式（含函数调用，如 `len(flows)`）；
- [ ] 会读调用堆栈，能通过点帧切换查看上下层变量；
- [ ] 会设条件断点，能写出 `flows[i] > 1500` 这类条件；
- [ ] 能用日志点在不改代码的前提下输出每一轮的值；
- [ ] 完成了案例 5A-1、5A-2 与练习 5A-1、5A-2 的全部修复，并能让程序输出正确结果。

## 前后章节衔接

- 本节只训练"工具"，语法范围**限定在第 1—5 章**；函数定义只作演示用，**第 6 章**将系统讲解 `def`、参数、返回值——届时回头看本节案例，`phase_green` 与 `avg_speed` 的形参错位会理解得更透。
- 本节反复用到的"先建空容器再追加""`get` 给默认值""`set` 去重计数"，分别对应**第 5 章**知识点 5.1、5.5、5.4。
- **第 9 章异常处理**将换一个角度排错：调试器用于"结果错但不报错"，异常处理用于"运行中会中断"——两者配合，才是一套完整的排错能力。
- **第 13 章综合项目**代码量上千行，届时"条件断点 + 调用堆栈"是定位问题的主要手段，本节练的就是那时的基本功。

---

[🏠 课程首页](../00_课程索引.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[◀ 上一章：列表元组集合与字典](../05列表元组集合与字典/05_列表元组集合与字典.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[下一章：函数 ▶](../06函数/06_函数.ipynb)